# Step 4 - LLM claim splitting, python-chess + Stockfish verification, taxonomy (pilot)

Pipeline per usable Step 3 explanation:

1. The `Verdict:` line is cut off and kept as its own claim; it is checked against the Stockfish label.
2. A **splitter LLM** (`config.SPLITTER_MODEL`, CPU, temperature 0, seed 46) cuts the rest into atomic claims. It sees only the text, copies words verbatim, and its output is rejected (rule-based fallback) if any quote is not a verbatim, in-order span or if the quotes cover < 80% of the text. Results are cached, so a re-run does not call the model again.
3. The embedded verifier checks each claim with **python-chess** (captures, attacks, defences, forks, pins, checks, castling, piece positions, legality of named moves/sequences) and with **Stockfish** (stored depth-18 evaluations, best move and lines from Step 2; optional live Stockfish for alternative moves named in the explanation).
4. The taxonomy category comes from the check that fired (if any), otherwise from the embedded keyword rules.
5. Every claim gets a **route**: `excluded_framing` / `auto_*` / `unverifiable_strategic` / `unverifiable_hedged` / `manual`. Only `manual` claims plus a blind 10% sample of auto-verified claims go to the annotation sheet; the sample measures how often the automatic checker is wrong.

Automatic verdicts are proposals until that sample shows their precision. The splitter and the checker are never allowed to see the condition for the decision, only the explanation text and (for the checker) the position and stored Stockfish results.
Save the notebook before **Run All**. Keep `RUN_SPLITTER = False` / `RUN_LIVE_STOCKFISH = False` for a dry run (cache only); set them in the run-control cell for the real run.

In [1]:
import datetime, hashlib, json, math, os, platform, random, re, subprocess, sys, time, urllib.error, urllib.request
from importlib import metadata
from pathlib import Path
import pandas as pd, chess, chess.engine
import config

PROJECT = Path.cwd()
MONTH = config.MONTH
MODELS = list(config.LLM_PILOT_MODELS)
CONDITIONS = list(config.LLM_PILOT_CONDITIONS)
REPEATS = int(config.LLM_PILOT_REPEATS_PER_CONDITION)
P1 = config.OUT_DIR / "step1_pilot" / MONTH
SF = config.OUT_DIR / "stockfish_recheck" / MONTH
S3 = config.OUT_DIR / "step3_llm_pilot" / MONTH
OUT = config.OUT_DIR / "step4_claims" / MONTH
OUT.mkdir(parents=True, exist_ok=True)
NOTEBOOK = "step4_claims.ipynb"
AUDIT_N = 12                        # explanations in the splitting audit sheet (checked by hand)
KEY = ["model", "condition", "pilot_id", "repeat"]
NUM_THREADS = max(1, (os.cpu_count() or 2) // 2)
OLLAMA_URL = config.LLM_PILOT_OLLAMA_URL
problems = []


def slug(m): return re.sub(r"[^A-Za-z0-9._-]", "_", m)
def warn(msg): problems.append(msg); print("FLAG", msg)
def sha256_file(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def csv_bytes(df): return df.to_csv(index=False, lineterminator="\n", float_format="%.10g").encode("utf-8")


def git(*a):
    try: return subprocess.run(["git", "-C", str(PROJECT), *a], check=True, capture_output=True, text=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError): return None


def notebook_code_sha256():
    nb = json.loads((PROJECT / NOTEBOOK).read_text(encoding="utf-8"))
    return hashlib.sha256("\n".join("".join(c["source"]) for c in nb["cells"] if "step4" in c.get("metadata", {}).get("tags", [])).encode()).hexdigest()


def helper_cell_hashes():
    nb = json.loads((PROJECT / NOTEBOOK).read_text(encoding="utf-8"))
    return {c["metadata"]["step4_helper"]: hashlib.sha256("".join(c["source"]).encode()).hexdigest()
            for c in nb["cells"] if c.get("metadata", {}).get("step4_helper")}


print(f"month {MONTH} | models {MODELS} | conditions {CONDITIONS} | repeats {REPEATS} | splitter {config.SPLITTER_MODEL}")


month 2026-10 | models ['qwen2.5:7b', 'deepseek-r1:8b', 'phi4-mini:latest', 'llama3.1:8b'] | conditions ['A', 'B', 'C'] | repeats 2 | splitter qwen2.5:7b


## Step 4 implementation helpers

These internal cells hold the segmentation/taxonomy, LLM splitting, and verification logic. They are defined before the workflow cells and need no separate Step 4 `.py` modules. Helper versions and source hashes are included in the manifest.

In [ ]:
import re, chess, chess.engine

# Step 4 manual-work reduction (deterministic, no LLM): subject resolution for fragments, prompt-echo checks, exchange-based
# checks (safe / sacrifice / loses a piece), move-origin checks and a main-clause polarity check. Only confident rules fire;
# everything else is left unchanged. All results are PROPOSALS audited through the blind audit sample.
# v2 adds: verdict lines, chat framing / cut-off fragments, intent ("prepares to ...") and vague claims routed as unverifiable,
# 'defended by' checks, hallucinated piece-on-square references, 'no change' in the stored evaluation.

REDUCE_VERSION = "reduce-v4"
_RD_VAL = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 100}
_RD_PC = r"(?P<p>king|queen|rook|bishop|knight|pawn)"
_RD_SQ = r"[a-h][1-8]"
_RD_COLOR = r"(?:(?P<c>white|black)(?:'s)?\s+)?"
_RD_PCS = {"king": chess.KING, "queen": chess.QUEEN, "rook": chess.ROOK, "bishop": chess.BISHOP, "knight": chess.KNIGHT, "pawn": chess.PAWN}
_RD_NEG = re.compile(r"\b(?:not|no|never|without|neither|nor|fails?|cannot|no longer)\b|n't", re.I)
_RD_MODAL = re.compile(r"\b(?:prepar\w+|aims?|could|might|may|potentially|possibly|perhaps|likely|threatens? to|plans? to|hopes?|if)\b", re.I)
_RD_AUX = re.compile(r"\b(?:also|now|indirectly|directly|currently|still|actively|particularly|especially|specifically|mainly|primarily|mostly|further|thereby)\s+", re.I)
_RD_ADJ = re.compile(r"\b(?:dark|light)-squared\s+|\b(?:queen|king)side\s+(?=(?:rook|bishop|knight|pawn)\b)", re.I)
_RD_GERUND = {"attacking": "attacks", "defending": "defends", "supporting": "supports", "protecting": "protects", "eyeing": "eyes",
              "targeting": "targets", "threatening": "threatens", "pressuring": "pressures", "capturing": "captures", "removing": "removes",
              "challenging": "challenges", "hitting": "hits"}
_RD_NOT_VERB = {"this", "its", "is", "was", "has", "as", "thus", "less", "across", "his", "us", "perhaps", "yes", "always", "sometimes"}
_RD_HAS_SUBJ = re.compile(r"^(?:it|this|the|a|an|his|its|black|white|he|she|[KQRBN]?[a-h]?x?[a-h][1-8][+#]?|(?:king|queen|rook|bishop|knight|pawn)s?)\b", re.I)
_RD_SENT_SUBJ = re.compile(r"^(?:however,?\s+|also,?\s+|additionally,?\s+)?(?:(?P<mv>it|this move|the move|the human(?:'s)? (?:move|player)|the human)\b|"
                           r"(?P<pc>(?:the\s+)?(?:rook|bishop|knight|queen|king|pawn)(?:\s+on\s+[a-h][1-8])?)\b|(?P<sq>[KQRBN]?[a-h]?x?[a-h][1-8][+#]?)\b)", re.I)


def rd_resolve(c):
    """Text used for verification: adds the missing subject to fragments ('supports the pawn on d6' -> 'It supports ...'),
    turns a leading gerund into a finite verb, drops filler adverbs and colour/side adjectives. The original claim_text is never changed."""
    t = c["claim_text"].strip()
    t = _RD_ADJ.sub("", _RD_AUX.sub("", t))
    frag = re.sub(r"^(?:and|but|while|which|however,?)\s+", "", t, flags=re.I)
    if _RD_HAS_SUBJ.match(frag):
        return frag
    w = frag.split(" ", 1)[0].lower()
    if w in _RD_NOT_VERB or not (w in _RD_GERUND or w.endswith("ing") or w.endswith("s")):
        return t
    subj = (c.get("source_subject") or "").strip()
    if not subj:
        m = _RD_SENT_SUBJ.match((c.get("sentence") or "").strip())
        if not m:
            return t
        subj = "It" if m.group("mv") else (m.group("pc") or m.group("sq"))
    if re.match(r"^(?:the\s+)?(?:move\b|human\b)", subj, re.I) or re.match(r"^(?:it|this move)$", subj, re.I):
        subj = "It"
    rest = frag.split(" ", 1)[1] if " " in frag else ""
    return f"{subj} {_RD_GERUND.get(w, w)} {rest}".strip()


def _rd_res(verdict, rule, detail="", source="python_chess", cat=None):
    return dict(auto_verdict=verdict, auto_rule=rule, auto_detail=detail, auto_source=source, check_category=cat)


_RD_HEADER = re.compile(r"^stockfish\s+(?:evaluation\s+(?:before|after)\s+the\s+move|principal\s+variation)\s*(?:\([^)]*)?:?$", re.I)


def rd_is_echo_header(text):
    """'Stockfish evaluation before the move (from White's point of view' = restated prompt text, not a claim."""
    return bool(_RD_HEADER.match(text.strip().rstrip(".")))


def rd_see(board, sq):
    """Net material the side to move wins by capturing on sq (least valuable attacker first, best stopping point, pins ignored)."""
    victim = board.piece_at(sq)
    attackers = board.attackers(board.turn, sq)
    if victim is None or not attackers:
        return 0
    frm = min(attackers, key=lambda s: _RD_VAL[board.piece_at(s).piece_type])
    b = board.copy(stack=False)
    b.push(chess.Move(frm, sq))
    return max(0, _RD_VAL[victim.piece_type] - rd_see(b, sq))


def _rd_net_after(board, move):
    """material result of the played move for the mover, counting the best recapture on the destination square."""
    after = board.copy(stack=False); after.push(move)
    got = 0
    if board.is_capture(move):
        v = chess.Piece(chess.PAWN, not board.turn) if board.is_en_passant(move) else board.piece_at(move.to_square)
        got = _RD_VAL[v.piece_type]
    return got - rd_see(after, move.to_square), after


def rd_verify_echo(c, ctx):
    """Claims that repeat the evidence given in the prompt: compare with the stored values (B and C only)."""
    text, sent = c["claim_text"], (c.get("sentence") or c["claim_text"])
    m = re.search(r"stockfish\s+evaluation\s+(?P<w>before|after)\s+the\s+move", sent, re.I)
    n = re.search(r"([-+−]?\d+(?:\.\d+)?)\s*cp", text, re.I)
    if m and n and (re.search(r"stockfish|point of view|positive", text, re.I)):
        want = ctx["eval_before_mover"] if m.group("w").lower() == "before" else ctx["eval_after_mover"]
        got = float(n.group(1).replace("−", "-"))
        ok = abs(got - want) < 0.5
        return _rd_res("correct" if ok else "incorrect", "echo_eval", f"stored {m.group('w').lower()} evaluation (mover view) {want:+d}, claim {got:+g}", "stockfish_stored", "Quality evaluation")
    if re.search(r"\b(?:no change|unchanged)\b", text, re.I) and re.search(r"stockfish|evaluation", text, re.I) and not re.search(r"\b(?:not|n't|but)\b", text, re.I):
        d = ctx["eval_after_mover"] - ctx["eval_before_mover"]
        return _rd_res("correct" if abs(d) <= 15 else "incorrect", "echo_unchanged", f"stored evaluation changes by {d:+d} cp", "stockfish_stored", "Quality evaluation")
    if re.match(r"^stockfish\s+principal\s+variation\s*:", text.strip(), re.I) and ctx.get("best_pv"):
        toks = [t.strip().rstrip(".") for t in text.split(":", 1)[1].split(",") if t.strip()]
        best = [x.rstrip("+#") for x in ctx["best_pv"]]
        ok = [t.rstrip("+#") for t in toks] == best[:len(toks)] and toks
        return _rd_res("correct" if ok else "incorrect", "echo_pv", "line equals the stored engine line" if ok else "line differs from the stored engine line", "stockfish_stored", "Tactical reasoning")
    return None


_RD_REF = re.compile(rf"(?:(?P<c>white|black)(?:'s)?\s+)?(?:the\s+)?(?:(?:dark|light)-squared\s+)?{_RD_PC}\s+(?:on|at)\s+(?P<s>{_RD_SQ})\b|\b(?P<s2>{_RD_SQ})[- ](?P<p2>king|queen|rook|bishop|knight|pawn|square)\b|\b(?P<p4>king|queen|rook|bishop|knight|pawn)\s*\((?P<s4>{_RD_SQ})\)", re.I)


def _rd_refs(text):
    """all 'white rook on f2' / 'the f6 pawn' style references in a text -> [(colour or None, piece type, square)] (squares excluded)."""
    out = []
    for m in _RD_REF.finditer(text):
        if m.group("p"):
            out.append((None if not m.group("c") else m.group("c").lower() == "white", _RD_PCS[m.group("p").lower()], chess.parse_square(m.group("s").lower())))
        elif m.group("p4"):
            out.append((None, _RD_PCS[m.group("p4").lower()], chess.parse_square(m.group("s4").lower())))
        elif m.group("p2").lower() != "square":
            out.append((None, _RD_PCS[m.group("p2").lower()], chess.parse_square(m.group("s2").lower())))
    return out


_RD_DEF = re.compile(rf"\b(?:is|was|are)\s+(?:(?P<adv>currently|previously|already|now|still)\s+)*(?:defended|protected|guarded|supported)\s+by\s+(?:(?P<c>white|black)(?:'s)?\s+)?(?:the\s+)?{_RD_PC}\s+on\s+(?P<s>{_RD_SQ})\b(?P<tail>.*)$", re.I)


def _rd_defended_by(c, t, board, after):
    m = _RD_DEF.search(t)
    if not m:
        return None
    b = board if re.search(r"\bpreviously\b|\bwas\b", m.group(0).split("by")[0], re.I) else after
    dsq = chess.parse_square(m.group("s").lower())
    dp = b.piece_at(dsq)
    if dp is None or dp.piece_type != _RD_PCS[m.group("p").lower()] or (m.group("c") and dp.color != (m.group("c").lower() == "white")):
        return _rd_res("incorrect", "defended_by", f"no such defender on {chess.square_name(dsq)}", cat="Board comprehension")
    # referent: last piece/square reference before the defender, in the claim itself, otherwise in the sentence before the claim
    head = t[:m.start()]
    refs = list(_RD_REF.finditer(head))
    if not refs:
        sent, orig = c.get("sentence") or "", c.get("claim_text_orig") or t
        i = sent.lower().find(orig.lower()[:30])
        refs = list(_RD_REF.finditer(sent[:i] if i > 0 else sent))
    if not refs:
        return None
    r = refs[-1]
    if r.group("p"):
        rsq, rpt = chess.parse_square(r.group("s").lower()), _RD_PCS[r.group("p").lower()]
    elif r.group("p4"):
        rsq, rpt = chess.parse_square(r.group("s4").lower()), _RD_PCS[r.group("p4").lower()]
    else:
        rsq, rpt = chess.parse_square(r.group("s2").lower()), (None if r.group("p2").lower() == "square" else _RD_PCS[r.group("p2").lower()])
    rp = b.piece_at(rsq)
    if rpt is not None and (rp is None or rp.piece_type != rpt):
        return _rd_res("incorrect", "defended_by", f"no {chess.piece_name(rpt)} on {chess.square_name(rsq)} to be defended", cat="Board comprehension")
    col = dp.color if rp is None else rp.color
    ok = dp.color == col and dsq in b.attackers(col, rsq)
    if ok and re.search(r"\b(?:and|but|so)\s+\w+", m.group("tail") or "", re.I):
        return None
    return _rd_res("correct" if ok else "incorrect", "defended_by", f"{chess.piece_name(dp.piece_type)} on {chess.square_name(dsq)} {'does' if ok else 'does not'} defend {chess.square_name(rsq)}", cat="Board comprehension")


_RD_CHAT = re.compile(r"^(?:okay|ok|sure|alright|certainly|great)\b[,!. ]|^let'?s\b|^here(?:'s| is)\b", re.I)


def rd_is_filler(text):
    """echo headers of the prompt, chat framing ('Okay, let's break down ...') and fragments cut off by the splitter ('The move Rfd8 (assuming it')."""
    t = text.strip()
    return rd_is_echo_header(t) or bool(_RD_CHAT.match(t)) or (t.count("(") > t.count(")") and len(t) < 45)


_RD_INTENT = re.compile(r"\b(?:prepar\w*|aim\w*|hop(?:e|es|ing)|plan\w*|intend\w*)\b", re.I)
_RD_VAGUE = re.compile(
    r"\bcounterplay\b|\binfluence\b|\b(?:future|potential)\s+(?:maneuvers?|moves?|operations?|plans?|attacks?|advances?|pushes?|kingside)\b|\buseful\s+for\b|\badvantageous\b"
    r"|\bdynamics\b|\bprospects\b|\binitiative\b|\bcomplexity\b|\bmore\s+dynamic\b|\b(?:somewhat|slightly)\s+\w+|\binto\s+play\b"
    r"|\b(?:central|powerful|passive|awkward|useless|aggressive)\s+(?:position|square)\b|\bleaving\s+the\s+\w+\s+pawn\b|\bgrim\b|\bequaliz\w+|\bsimplif\w+|\bdisrupt\w*|\bconcessions?\b|\bdecisive\b|\bdevastating\b|\bstructure\b|\bpressur\w+|\bsolid\b", re.I)


def rd_is_intent(text):
    """'prepares to ...' / 'aims to ...': a plan, not a fact on the board; routed as unverifiable (hedged)."""
    return bool(_RD_INTENT.search(text))


def rd_is_vague(text):
    """positional or purely general wording that no board check can decide; routed as unverifiable (strategic)."""
    return bool(_RD_VAGUE.search(text))


def rd_verify_verdict_text(text, ctx):
    """'Verdict: good' / 'Verdict: bad' that the splitter returned as a claim."""
    m = re.match(r"^verdict\s*:\s*(good|bad)\.?$", text.strip(), re.I)
    if not m or ctx.get("label") not in ("good", "bad"):
        return None
    ok = m.group(1).lower() == ctx["label"]
    return _rd_res("correct" if ok else "incorrect", "verdict_vs_label", f"verdict {m.group(1).lower()}, Stockfish label {ctx['label']}", "label", "Quality evaluation")


def _rd_sides(text, mover):
    named = {x.lower() for x in re.findall(r"\b(white|black)(?:'s)?\b", text, re.I)}
    return named


def _rd_null_score(engine, b, mover):
    """Stockfish score (mover's view, cp) of position b, cached in the live-Stockfish cache so reruns need no engine."""
    key = f"{engine.key_prefix}|null|{b.fen()}"
    if key not in engine.cache:
        cp = engine.engine.analyse(b, chess.engine.Limit(depth=engine.depth))["score"].pov(mover).score(mate_score=10000)
        engine.cache[key] = int(max(-10000, min(10000, cp)))
        if engine.cache_path:
            import json
            with engine.cache_path.open("a", encoding="utf-8", newline="\n") as f:
                f.write(json.dumps([key, engine.cache[key]]) + "\n")
    return engine.cache[key]


def rd_threat_exists(board, move, after, mover, engine=None):
    """Does the mover threaten something after the move? Null-move test: if the opponent passed, could the mover (a) mate, (b) win material
    (exchange count >= 2) or (c) -- with live Stockfish -- improve the evaluation by >= 100 cp. The opponent being in check counts as a threat.
    Returns (True/False/None, detail); None = python-chess found nothing and no engine was available, so nothing can be said."""
    if after.is_check():
        return True, "the move gives check"
    b = after.copy(stack=False); b.push(chess.Move.null())
    for mv in b.legal_moves:
        b2 = b.copy(stack=False); b2.push(mv)
        if b2.is_checkmate():
            return True, f"mate threat ({b.san(mv)})"
    for mv in b.legal_moves:
        if b.is_capture(mv) and not b.is_en_passant(mv):
            gain = _RD_VAL[b.piece_at(mv.to_square).piece_type]
            b2 = b.copy(stack=False); b2.push(mv)
            if gain - rd_see(b2, mv.to_square) >= 2:
                return True, f"{b.san(mv)} would win about {gain - rd_see(b2, mv.to_square)} points of material"
    if engine is not None:
        try:
            gain = _rd_null_score(engine, b, mover) - engine.score_after(board.fen(), move.uci())
        except Exception as e:
            return None, f"engine error: {e}"
        if gain >= 100:
            return True, f"Stockfish: evaluation would rise by {gain:+d} cp if the opponent passed"
        if gain < 30:
            return False, f"Stockfish: no real threat (evaluation change {gain:+d} cp if the opponent passed)"
    return None, "no threat found by python-chess"


_RD_TH_ADJ = r"(?:(?:immediate|direct|clear|tactical|real|serious|potential|pinning)\s+)*"
_RD_THREAT_POS = re.compile(rf"^(?:(?:and|however|also),?\s+)*(?:(?:it|this\s+move|the\s+move|this)\s+)?(?:also\s+)?(?:creat\w+|pos\w+|mak\w+)\s+(?:(?:a|an|any|some)\s+)?{_RD_TH_ADJ}threats?\.?$", re.I)
_RD_THREAT_NEG = re.compile(rf"^(?:(?:and|however|but|as),?\s+)*(?:(?:it|this\s+move|the\s+move)\s+)?(?:does\s?not|doesn't|did\s+not|didn't|fails?\s+to)\s+(?:immediately\s+|directly\s+)?(?:creat\w*|pos\w*|mak\w*)\s+(?:(?:a|an|any)\s+)?{_RD_TH_ADJ}threats?\.?$", re.I)


def rd_verify_threat(c, board, move, after, mover, engine=None):
    """Only the plain generic wording: 'it creates (immediate) threats' / 'it does not create any threats' (nothing specific named)."""
    t = c["claim_text"].replace("\u2019", "'").strip()
    neg, pos = bool(_RD_THREAT_NEG.match(t)), bool(_RD_THREAT_POS.match(t))
    if not (neg or pos):
        return None
    found, detail = rd_threat_exists(board, move, after, mover, engine)
    if found is None:
        return None
    ok = found if pos else not found
    return _rd_res("correct" if ok else "incorrect", "threat_null_move", detail, "stockfish_live" if detail.startswith("Stockfish") else "python_chess", "Tactical reasoning")


def rd_verify(c, board, after, move, mover, ctx, engine=None):
    t = c["claim_text"].replace("’", "'")
    moved = board.piece_at(move.from_square)
    name = lambda m: chess.piece_name(m.piece_type)

    # 0. 'does not (significantly) improve the position': decided with the stored evaluation change (mover's view)
    m = re.search(r"\b(?:does\s+not|doesn't|did\s+not|didn't|fails?\s+to|not)\s+(?:significantly\s+|really\s+|substantially\s+|meaningfully\s+)?improv\w+\s+(?:the\s+|(?P<side>white|black)'s\s+)?position\b", t, re.I)
    r = rd_verify_threat(c, board, move, after, mover, engine)
    if r is not None:
        return r
    if m and not re.search(r"\b(?:but|although|though)\b", t, re.I) and (not m.group("side") or (m.group("side").lower() == "white") == mover):
        d = ctx["eval_after_mover"] - ctx["eval_before_mover"]
        tail_extra = bool(re.match(r"\s*(?:,|or\b|and\b|as\b|because\b|since\b)", t[m.end():], re.I))   # a second part we do not check
        if (d < 50 and not tail_extra) or d >= 100:
            return _rd_res("correct" if d < 50 else "incorrect", "no_improvement", f"stored evaluation change for the mover {d:+d} cp", "stockfish_stored", "Quality evaluation")

    if _RD_NEG.search(t):
        return None

    # 1. 'The bishop on b1 is now on e4' (also 'moves from b1 to e4' is handled elsewhere)
    m = re.search(rf"\b(?:the\s+)?{_RD_COLOR}{_RD_PC}\s+on\s+(?P<a>{_RD_SQ})\s+(?:is|has moved|moves|moved|goes|went)\s+(?:now\s+)?(?:on|at|to)\s+(?P<b>{_RD_SQ})\b", t, re.I)
    if m:
        a, b, pt = chess.parse_square(m.group("a").lower()), chess.parse_square(m.group("b").lower()), _RD_PCS[m.group("p").lower()]
        ok = move.from_square == a and move.to_square == b and moved.piece_type == pt
        return _rd_res("correct" if ok else "incorrect", "moved_from_to", f"played: {name(moved)} {chess.square_name(move.from_square)}-{chess.square_name(move.to_square)}", cat="Board comprehension")

    # 1a. '<piece> from a1 to b1' / 'moving it to b1' / 'the rook on f1 now occupies the square f4' / 'a square on the back rank'
    m = (re.search(rf"\b{_RD_PC}\s+from\s+(?P<a>{_RD_SQ})\s+to\s+(?P<b>{_RD_SQ})\b", t, re.I)
         or re.search(rf"\b{_RD_PC}\s+on\s+(?P<a>{_RD_SQ})\s+(?:now\s+)?(?:occupies|takes|captures|reaches|lands\s+on|arrives\s+(?:on|at))\s+(?:the\s+square\s+)?(?P<b>{_RD_SQ})\b", t, re.I))
    if m:
        a, b, pt = chess.parse_square(m.group("a").lower()), chess.parse_square(m.group("b").lower()), _RD_PCS[m.group("p").lower()]
        ok = move.from_square == a and move.to_square == b and moved.piece_type == pt
        return _rd_res("correct" if ok else "incorrect", "moved_from_to", f"played: {name(moved)} {chess.square_name(move.from_square)}-{chess.square_name(move.to_square)}", cat="Board comprehension")
    m = re.search(rf"\bmoving\s+(?:it|the\s+{_RD_PC})\s+to\s+(?P<b>{_RD_SQ})\b", t, re.I)
    if m and (not m.group("p") or _RD_PCS[m.group("p").lower()] == moved.piece_type):
        ok = move.to_square == chess.parse_square(m.group("b").lower())
        return _rd_res("correct" if ok else "incorrect", "moved_to", f"the played move goes to {chess.square_name(move.to_square)}", cat="Board comprehension")
    m = re.search(r"\boccupies\s+a\s+square\s+on\s+the\s+(?P<r>back|first|eighth)\s+rank\b", t, re.I)
    if m and moved.piece_type != chess.PAWN:
        want = chess.square_rank(move.to_square) == ({"back": 0 if mover == chess.WHITE else 7, "first": 0, "eighth": 7}[m.group("r").lower()])
        return _rd_res("correct" if want else "incorrect", "moved_to_rank", f"the played move goes to {chess.square_name(move.to_square)}", cat="Board comprehension")

    # 1a2. 'removes a defender from the e5 pawn': a defender of that piece before the move is gone after it
    m = re.search(rf"\b(?:removes?|removing|eliminates?)\s+a\s+defender\s+(?:from|of)\s+(?:the\s+){_RD_COLOR}(?:(?P<p2>king|queen|rook|bishop|knight|pawn)\s+on\s+(?P<s>{_RD_SQ})|(?P<s3>{_RD_SQ})\s+(?P<p3>king|queen|rook|bishop|knight|pawn))\b", t, re.I)
    if m:
        sq = chess.parse_square((m.group("s") or m.group("s3")).lower()); pt = _RD_PCS[(m.group("p2") or m.group("p3")).lower()]
        tp = board.piece_at(sq)
        if tp is not None and tp.piece_type == pt and tp.color != mover:
            lost = set(board.attackers(tp.color, sq)) - set(after.attackers(tp.color, sq)) if after.piece_at(sq) is not None else set(board.attackers(tp.color, sq))
            return _rd_res("correct" if lost else "incorrect", "removes_defender", f"defenders of {chess.square_name(sq)} before/after: {len(board.attackers(tp.color, sq))}/{len(after.attackers(tp.color, sq))}", cat="Tactical reasoning")

    # 1a3. 'relocates the bishop from the kingside to the queenside' / 'reduces Black's material' / 'this move defends the g7 pawn'
    m = re.search(r"\bfrom\s+the\s+(?P<a>king|queen)side\s+to\s+the\s+(?P<b>king|queen)side\b", t, re.I)
    if m and m.group("a").lower() != m.group("b").lower():
        side = lambda sq: "king" if chess.square_file(sq) >= 4 else "queen"
        ok = side(move.from_square) == m.group("a").lower() and side(move.to_square) == m.group("b").lower()
        return _rd_res("correct" if ok else "incorrect", "moved_side", f"played: {chess.square_name(move.from_square)}-{chess.square_name(move.to_square)}", cat="Board comprehension")
    m = re.search(r"\b(?:reduc\w+|lower\w*|decreas\w+)\s+(?:(?P<c>white|black)'s|the\s+opponent's)\s+material\b", t, re.I)
    if m and (not m.group("c") or (m.group("c").lower() == "white") != mover):
        ok = board.is_capture(move)
        return _rd_res("correct" if ok else "incorrect", "reduces_material", "the played move captures a piece" if ok else "the played move is not a capture", cat="Tactical reasoning")
    m = re.match(rf"^(?:it|this\s+move|the\s+move)\s+(?:also\s+)?(?:defends|protects|guards|supports)\s+the\s+{_RD_COLOR}(?:(?P<p2>king|queen|rook|bishop|knight|pawn)\s+on\s+(?P<s>{_RD_SQ})|(?P<s3>{_RD_SQ})\s+(?P<p3>king|queen|rook|bishop|knight|pawn))\.?$", t, re.I)
    if m:
        sq = chess.parse_square((m.group("s") or m.group("s3")).lower()); tp = after.piece_at(sq)
        if tp is not None and tp.color == mover and tp.piece_type == _RD_PCS[(m.group("p2") or m.group("p3")).lower()]:
            if move.to_square in after.attackers(mover, sq):
                return _rd_res("correct", "move_defends", f"the moved piece on {chess.square_name(move.to_square)} defends {chess.square_name(sq)}", cat="Tactical reasoning")
            if set(after.attackers(mover, sq)) <= set(board.attackers(mover, sq)) - {move.from_square}:
                return _rd_res("incorrect", "move_defends", f"{chess.square_name(sq)} has no new defender after the move", cat="Tactical reasoning")

    # 1b. 'X is (currently / previously) defended by the Y on sq': Y must be there and really defend X
    r = _rd_defended_by(c, t, board, after)
    if r is not None:
        return r

    # 1c. a named '<colour> <piece> on <square>' that is on that square neither before nor after the move does not exist
    if not re.search(r"\b(?:if|when|could|would|might)\b", t, re.I):
        for col, pc, sq in _rd_refs(t):
            if not any(b.piece_at(sq) is not None and b.piece_at(sq).piece_type == pc and (col is None or b.piece_at(sq).color == col) for b in (board, after)):
                return _rd_res("incorrect", "piece_not_there", f"no {'' if col is None else ('white ' if col else 'black ')}{chess.piece_name(pc)} on {chess.square_name(sq)} before or after the move", cat="Board comprehension")

    if _RD_MODAL.search(t):
        return None

    # 2. removes / captures (also passive): the played move must capture exactly that piece
    m = (re.search(rf"\b(?:removes?|removing|eliminates?)\s+(?:the|a|an)\s+{_RD_COLOR}{_RD_PC}\b(?!\s*'s|\s+from\s+(?!the\s+board)|\s+protect|\s+defen)", t, re.I)
         or re.search(rf"\b(?:the|a)\s+{_RD_COLOR}{_RD_PC}\s+(?:is|was)\s+(?:removed|captured|taken|eliminated)\b", t, re.I))
    if m:
        if not board.is_capture(move):
            return _rd_res("incorrect", "removes_piece", "the played move is not a capture", cat="Tactical reasoning")
        victim = chess.Piece(chess.PAWN, not mover) if board.is_en_passant(move) else board.piece_at(move.to_square)
        ok = victim.piece_type == _RD_PCS[m.group("p").lower()] and (not m.group("c") or victim.color == (m.group("c").lower() == "white"))
        return _rd_res("correct" if ok else "incorrect", "removes_piece", f"the played move captures a {chess.piece_name(victim.piece_type)}", cat="Tactical reasoning")

    # 3. '<piece> on f2 is safe'
    m = re.search(rf"\b(?:the\s+)?{_RD_COLOR}{_RD_PC}\s+(?:on\s+(?P<s>{_RD_SQ})\s+)?(?:is|are|remains?)\s+(?:now\s+)?(?:safe|secure|protected)\b", t, re.I)
    if m and m.group("s"):
        sq = chess.parse_square(m.group("s").lower())
        p = after.piece_at(sq)
        if p is None or p.piece_type != _RD_PCS[m.group("p").lower()]:
            return _rd_res("incorrect", "piece_safe", f"no such piece on {chess.square_name(sq)} after the move", cat="Tactical reasoning")
        if p.color != mover:
            return None
        if not after.is_attacked_by(not mover, sq):
            return _rd_res("correct", "piece_safe", "not attacked after the move", cat="Tactical reasoning")
        loss = rd_see(after, sq)
        if loss >= 2:
            return _rd_res("incorrect", "piece_safe", f"opponent wins about {loss} points of material there (exchange count)", cat="Tactical reasoning")
        if loss == 0:
            return _rd_res("correct", "piece_safe", "attacked but the exchange does not lose material", cat="Tactical reasoning")
        return None

    # 4. 'sacrifices the rook' / 'loses the rook': only a confident confirmation is decided
    m = re.search(rf"\b(?:sacrific\w+|loses?)\s+(?:the|a|an|his|its)?\s*{_RD_COLOR}{_RD_PC}\b", t, re.I)
    if m and (not m.group("c") or (m.group("c").lower() == "white") == mover) and moved.piece_type == _RD_PCS[m.group("p").lower()]:
        net, _ = _rd_net_after(board, move)
        if net <= -2:
            return _rd_res("correct", "piece_lost", f"the moved {name(moved)} is lost in the exchange (net {net:+d})", cat="Tactical reasoning")
        mf = re.search(r"\bfor\s+(?P<n>a|one|two|three|the)\s+(?P<q>pawns?|knight|bishop|rook|queen)\b", t, re.I)
        if mf and board.is_capture(move) and not board.is_en_passant(move):
            want = {"a": 1, "one": 1, "two": 2, "three": 3, "the": 1}[mf.group("n").lower()] * {"pawn": 1, "knight": 3, "bishop": 3, "rook": 5, "queen": 9}[mf.group("q").lower().rstrip("s")]
            got = _RD_VAL[board.piece_at(move.to_square).piece_type]
            return _rd_res("correct" if want == got else "incorrect", "sacrifice_for", f"the move captures material worth {got}, the claim says {want}", cat="Tactical reasoning")
        if net >= 1 and not re.search(r"\b(?:for\s+(?:a|two|the)\b|compensation|initiative|attack)", t, re.I):
            return _rd_res("incorrect", "piece_lost", f"the moved {name(moved)} is not lost in the exchange (net {net:+d})", cat="Tactical reasoning")
        return None

    # 5. polarity of the main clause ('this move significantly worsens White's position as it allows ...')
    if re.match(r"^(?:it|this move|the move|the human(?:'s)? move)\b", t, re.I) or re.search(r"\bthis move\b", t, re.I):
        label = ctx.get("label")
        main = re.split(r"\s+(?:as|because|since|which|allowing|allows|leading|resulting|while|by)\b|,", t, maxsplit=1, flags=re.I)[0]
        pos = bool(re.search(r"\b(?:good|strong|best|excellent|solid|sound|improves?|strengthens?|favou?rable|effective|accurate)\b", main, re.I))
        neg = bool(re.search(r"\b(?:bad|weak|poor|blunder\w*|mistake\w*|inaccura\w*|dubious|worse|worsens?|losing|weakens?|risky|unfavou?rable|questionable)\b", main, re.I))
        named = _rd_sides(main, mover)
        if label in ("good", "bad") and pos != neg:
            if named and named == {"black" if mover == chess.WHITE else "white"}:
                pos, neg = neg, pos
            ok = (pos and label == "good") or (neg and label == "bad")
            return _rd_res("correct" if ok else "incorrect", "polarity_main_clause", f"claim {'positive' if pos else 'negative'}, Stockfish label {label}", "label", "Quality evaluation")
    return None


In [2]:
# Claim segmentation, taxonomy and deterministic pre-checks.

LIB_VERSION = "claims-v5"
CATEGORIES = ("Board comprehension", "Legality", "Quality evaluation", "Tactical reasoning", "Other")
SQ = r"[a-h][1-8]"
PIECE = r"(king|queen|rook|bishop|knight|pawn)"
PIECE_TYPES = {"king": chess.KING, "queen": chess.QUEEN, "rook": chess.ROOK, "bishop": chess.BISHOP,
               "knight": chess.KNIGHT, "pawn": chess.PAWN}
COLORS = {"white": chess.WHITE, "black": chess.BLACK}

# ---------------------------------------------------------------- segmentation
_ABBREV = re.compile(r"\b(e\.g|i\.e|vs|etc|cf)\.", re.I)
_LIST_MARK = re.compile(r"(?m)^\s*(?:\d+[.)]|[-•*])\s+")
_MD = re.compile(r"[*_`#]+")
_LIB_FRAMING = re.compile(r"^(?:a|the)\s+(?:human|player|human player)\s+(?:might|may|could|would|can)?\s*(?:have\s+)?"
                      r"(?:choose|chosen|play|played|opt for|prefer)s?\s+(?:this move|\S+)\s+(?:because|since|as)\s+(?:that\s+)?", re.I)
_LEAD = re.compile(r"^(?:overall|additionally|furthermore|moreover|also|however|finally|lastly|in addition|thus|therefore|"
                   r"and|but|then),?\s+", re.I)
_SPLIT_PUNCT = re.compile(r"\s*;\s*|\s+[—–]\s+|\s+-\s+")
_SPLIT_CONN = re.compile(r",\s+(?=(?:but|while|whereas|which|so that)\b|[a-z]+ing\b)", re.I)   # ', which ...', ', capturing ...'
_BY_ING = re.compile(r"^(by\s+[a-z]+ing\b[^,]*?),\s+(.+)$", re.I)                              # 'By moving the bishop to g6, Black ...'
_SPLIT_REASON = re.compile(r"\s+(?=(?:because|since|so that|whereas|indicates?|suggests?|shows?|means|implies)\b)", re.I)
_AND = re.compile(r"(,?\s+and\s+)", re.I)
_VERBS = re.compile(r"\b(?:is|are|was|were|has|have|had|can|could|will|would|might|may|does|do|opens?|attacks?|threatens?|puts?|"
                    r"controls?|creates?|prepares?|allows?|forces?|gains?|wins?|loses?|captures?|takes?|exchanges?|trades?|"
                    r"develops?|defends?|protects?|pins?|forks?|improves?|weakens?|strengthens?|activates?|centralizes?|"
                    r"removes?|eliminates?|gives?|blocks?|supports?|pressures?|targets?|moves?|leaves?|keeps?|maintains?|"
                    r"increases?|reduces?|exposes?|restricts?|secures?|aims?|seeks?|intends?|ensures?|enables?|gets?|makes?|"
                    r"forms?|sets?|leads?|provides?|offers?|brings?|places?|plays?|threatening|attacking)\b", re.I)
_HEDGE = re.compile(r"\b(might|may|could|potentially|possibly|perhaps|likely|aims? to|seeks? to|intends?|hopes?|plans? to)\b", re.I)


def split_sentences(text):
    t = _MD.sub("", _LIST_MARK.sub("", text.replace("’", "'")))
    t = re.sub(r"\s+", " ", t).strip()
    t = _ABBREV.sub(lambda m: m.group(0).replace(".", "§"), t)
    return [p.replace("§", ".").strip() for p in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])", t) if p.strip()]


def _split_and(clause):
    """split at 'and' only when both sides contain a verb (so 'knight and bishop' stays together)."""
    parts = _AND.split(clause)
    out, cur = [], parts[0]
    for i in range(1, len(parts), 2):
        right = parts[i + 1]
        if _VERBS.search(cur) and _VERBS.search(right):
            out.append(cur); cur = right
        else:
            cur += parts[i] + right
    out.append(cur)
    return out


def split_clauses(sentence):
    s = _LIB_FRAMING.sub("", sentence)
    s = _LEAD.sub("", s)
    m = _BY_ING.match(s)
    pieces = [m.group(1), m.group(2)] if m else [s]
    for rx in (_SPLIT_PUNCT, _SPLIT_CONN, _SPLIT_REASON):
        pieces = [q for p in pieces for q in rx.split(p)]
    pieces = [q for p in pieces for q in _split_and(p)]
    out = []
    for p in pieces:
        p = _LEAD.sub("", p.strip(" ,.;:"))
        if len(p.split()) >= 2:
            out.append(p)
    return out


def extract_claims(text):
    """-> list of dict(sentence_idx, clause_idx, sentence, claim_text, hedged)"""
    rows = []
    for si, sent in enumerate(split_sentences(text)):
        for ci, clause in enumerate(split_clauses(sent)):
            if norm(clause) not in norm(sent):
                raise ValueError(f"Rule splitter produced text not found in its source sentence: {clause[:80]}")
            rows.append(dict(sentence_idx=si, clause_idx=ci, sentence=sent, claim_text=clause,
                             source_quote="", source_subject="",
                             hedged=bool(_HEDGE.search(clause))))
    return rows


# ---------------------------------------------------------------- taxonomy
_ON = rf"(?:the\s+)?(?:(white|black)(?:'s)?\s+)?{PIECE}s?\s+(?:currently\s+|now\s+)?(?:(?:is|are|sits|stands|remains)\s+)?(?:on|at)\s+({SQ})\b"
PIECE_ON_SQ = re.compile(_ON, re.I)
_SQUARE_PIECE = re.compile(rf"\b(?:(white|black)(?:'s)?\s+)?({SQ})\s+{PIECE}s?\b", re.I)
PRECEDENCE = ("Legality", "Tactical reasoning", "Quality evaluation", "Board comprehension")
RULES = [
    ("Legality", "L1", r"\b(illegal|legal|not allowed|allowed to|cannot|can't|can castle|can move|possible move)\b"),
    ("Tactical reasoning", "T1", r"\b(forks?|forking|pins?|pinned|pinning|skewer\w*|discovered|x-ray)\b"),
    ("Tactical reasoning", "T2", r"\b(attack\w*|threat\w*|defen[cd]\w*|protect\w*|hanging|undefended|en prise)\b"),
    ("Tactical reasoning", "T3", r"\b(captur\w*|takes|taking|recaptur\w*|exchang\w*|trad\w*|sacrific\w*|material|check\w*|mate|forc\w*"
                                  r"|wins? (?:a|an|the)|loses? (?:a|an|the))\b"),
    ("Quality evaluation", "Q1", r"\b(good|bad|strong|weak|best|worst|blunder\w*|mistake\w*|inaccura\w*|excellent|poor|dubious|winning|"
                                  r"losing|decisive|advantage\w*|disadvantage\w*|better|worse|improv\w*|weaken\w*|strengthen\w*|solid|"
                                  r"risky|sound|favou?rable|unfavou?rable|equal|balanced|evaluat\w*|centipawns?|cp|stockfish|engine)\b"),
    ("Board comprehension", "B1", _ON),
    ("Board comprehension", "B2", r"\b(doubled|isolated|passed|backward) pawns?\b|\bpawn (structure|chain)\b|"
                                   r"\b(sits|stands|remains|located|positioned|placed|occupi\w*) (on|at)\b"),
    ("Board comprehension", "B3", rf"\b{SQ}\b"),
]
_RULES_C = [(c, i, re.compile(p, re.I)) for c, i, p in RULES]
_STRATEGIC = re.compile(r"\b(control\w*|cent(?:er|re)\w*|activ\w*|develop\w*|initiative|space|coordinat\w*|king safety|pressure|"
                        r"mobility|tempo|plan\w*|strateg\w*|weakness\w*|outpost\w*|open\w*|file|diagonal|flexib\w*)\b", re.I)
_ENGINE_WORDS = re.compile(r"\b(stockfish|engine|centipawns?|cp|principal variation|evaluation)\b", re.I)


def classify(claim):
    hits = [(c, i) for c, i, rx in _RULES_C if rx.search(claim)]
    cats = {c for c, _ in hits}
    cat = next((c for c in PRECEDENCE if c in cats), "Other")
    rule = next((i for c, i in hits if c == cat), "default")
    strategic = bool(_STRATEGIC.search(claim))
    review = len(cats) > 1 or (cat == "Other" and not strategic)
    return dict(category=cat, category_rule=rule, also_matches="|".join(sorted(cats - {cat})),
                is_strategic=strategic, review_category=review, mentions_engine=bool(_ENGINE_WORDS.search(claim)))


# ---------------------------------------------------------------- programmatic pre-checks (proposals)
_NEG = re.compile(r"\b(not|no|never|without|neither|nor|fails?|cannot|can't|doesn't|isn't|won't)\b|n't", re.I)
_MODAL = re.compile(r"(threat\w*|could|can|will|would|might|may|allow\w*|prevent\w*|to|aim\w*|seek\w*|able|before|if)\s+(?:\w+\s+){0,2}$", re.I)
_CAPTURE = re.compile(
    rf"\b(?:captur\w+|takes?|taking)\s+(?:the\s+|a\s+|an\s+)?"
    rf"(?:(?P<color>white|black)(?:'s)?\s+)?(?:(?P<square_before>{SQ})\s+)?"
    r"(?P<piece>king|queen|rook|bishop|knight|pawn)\b"
    rf"(?:\s+(?:on|at)\s+(?P<square_after>{SQ}))?", re.I)
_CAPTURE_ACTOR = re.compile(rf"\b(?:the\s+)?(?:(?P<color>white|black)(?:'s)?\s+)?(?P<piece>king|queen|rook|bishop|knight|pawn)\s+(?:on|at)\s+(?P<square>{SQ})\s+captur\w*\b", re.I)
_MOVE_ROUTE = re.compile(rf"\b(?:mov(?:es?|ing)|plac(?:es?|ing)|putting|puts?|develop(?:s|ing)?|retreat(?:s|ing)?|bring(?:s|ing)?|advanc(?:es?|ing))\s+(?:the\s+|a\s+|an\s+)?(?:(?P<color>white|black)(?:'s)?\s+)?(?P<piece>king|queen|rook|bishop|knight|pawn)\s+from\s+(?P<origin>{SQ})\s+to\s+(?P<destination>{SQ})\b", re.I)
_MOVES = re.compile(rf"\b(?:mov(?:es?|ing)|plac(?:es?|ing)|putting|puts?|develop(?:s|ing)?|retreat(?:s|ing)?|bring(?:s|ing)?|advanc(?:es?|ing))\s+(?:the\s+)?(?:(white|black)(?:'s)?\s+)?{PIECE}\s+(?:\w+\s+)?(?:to|on)\s+({SQ})\b", re.I)
_CHECK = re.compile(r"\b(?:gives?|delivers?|puts?|places?)\b[^.]{0,25}\bcheck\b|\bwith check\b|\bchecks the\b", re.I)
_INTERPRET = re.compile(r"\b(indicat\w*|mean\w*|suggest\w*|show\w*|impl\w*|reflect\w*|therefore|because)\b", re.I)
_CPNUM = re.compile(r"([-+−]?\d+(?:\.\d+)?)\s*(?:cp|centipawns?)", re.I)
_POS = re.compile(r"\b(good|strong|best|excellent|solid|sound|improv\w*|strengthen\w*|advantage|better|winning|favou?rable|effective|accurate)\b", re.I)
_NEGW = re.compile(r"\b(bad|weak|poor|blunder\w*|mistake\w*|inaccura\w*|dubious|worse|losing|weaken\w*|risky|disadvantage|unfavou?rable|questionable|loses)\b", re.I)


def _lib_res(verdict, rule, detail=""):
    return dict(auto_verdict=verdict, auto_rule=rule, auto_detail=detail)


def precheck(claim, board, move, label=None, provided_evals=()):
    """board = position BEFORE the move, move = the played move. verdict in correct / incorrect / unknown (+ rule, detail)."""
    c = claim.replace("’", "'")
    if _NEG.search(c):
        return _lib_res("unknown", "negation_unsupported")
    after = board.copy(); after.push(move)
    mover = board.turn

    actor = _CAPTURE_ACTOR.search(c)
    if actor and not _MODAL.search(c[:actor.start()][-30:]):
        actor_piece, actor_square = actor.group("piece").lower(), actor.group("square").lower()
        square = chess.parse_square(actor_square)
        actual = board.piece_at(square)
        actual_desc = "nothing" if actual is None else f"a {'white' if actual.color == chess.WHITE else 'black'} {chess.piece_name(actual.piece_type)}"
        color = actor.group("color")
        expected = f"{color.lower() + ' ' if color else ''}{actor_piece}"
        if actual is None or actual.piece_type != PIECE_TYPES[actor_piece] or (color and COLORS[color.lower()] != actual.color):
            return _lib_res("incorrect", "piece_on_square", f"{actor_square} holds {actual_desc} before the move, not the claimed {expected}")
        if square != move.from_square:
            return _lib_res("incorrect", "moved_piece", f"played {actor_piece} moves from {chess.square_name(move.from_square)}, not {actor_square}")

    m = _CAPTURE.search(c)
    if m and not _MODAL.search(c[:m.start()][-30:]):
        color, piece = m.group("color"), m.group("piece").lower()
        square_before, square_after = m.group("square_before"), m.group("square_after")
        if not board.is_capture(move):
            return _lib_res("incorrect", "capture_identity", "the played move is not a capture")
        victim = chess.Piece(chess.PAWN, not mover) if board.is_en_passant(move) else board.piece_at(move.to_square)
        if victim is None:
            return _lib_res("incorrect", "capture_identity", "no captured piece is present on the destination square")
        if square_before and square_after and square_before.lower() != square_after.lower():
            return _lib_res("incorrect", "capture_identity", f"claim gives conflicting capture squares: {square_before} and {square_after}")
        sq = square_before or square_after
        if sq and chess.parse_square(sq.lower()) != move.to_square:
            return _lib_res("incorrect", "capture_identity", f"capture happens on {chess.square_name(move.to_square)}, not {sq}")
        if PIECE_TYPES[piece] != victim.piece_type:
            return _lib_res("incorrect", "capture_identity", f"captures a {chess.piece_name(victim.piece_type)} on {chess.square_name(move.to_square)}, not a {piece}")
        if color and COLORS[color.lower()] != victim.color:
            return _lib_res("incorrect", "capture_identity", "captured piece has the other colour")
        return _lib_res("correct", "capture_identity", f"captures a {piece} on {chess.square_name(move.to_square)}")

    m = _MOVE_ROUTE.search(c)
    if m:
        piece = m.group("piece").lower()
        moved = board.piece_at(move.from_square)
        actual_from, actual_to = chess.square_name(move.from_square), chess.square_name(move.to_square)
        claimed_from, claimed_to = m.group("origin").lower(), m.group("destination").lower()
        if moved.piece_type != PIECE_TYPES[piece]:
            return _lib_res("incorrect", "moved_piece", f"played move moves a {chess.piece_name(moved.piece_type)}, not a {piece}")
        color = m.group("color")
        if color and COLORS[color.lower()] != moved.color:
            return _lib_res("incorrect", "moved_piece", f"played piece is {'white' if moved.color == chess.WHITE else 'black'}, not {color.lower()}")
        if claimed_from != actual_from:
            return _lib_res("incorrect", "moved_piece", f"{piece} moves from {actual_from}, not {claimed_from}")
        if claimed_to != actual_to:
            return _lib_res("incorrect", "moved_piece", f"{piece} moves to {actual_to}, not {claimed_to}")
        return _lib_res("correct", "moved_piece", f"{piece} moves from {actual_from} to {actual_to}")

    m = _MOVES.search(c)
    if m:
        piece, sq = m.group(2).lower(), chess.parse_square(m.group(3).lower())
        moved = board.piece_at(move.from_square)
        ok = moved.piece_type == PIECE_TYPES[piece] and move.to_square == sq
        return _lib_res("correct" if ok else "incorrect", "moved_piece",
                    f"played: {chess.piece_name(moved.piece_type)} to {chess.square_name(move.to_square)}")

    m = PIECE_ON_SQ.search(c)
    square_first = False
    if not m:
        m = _SQUARE_PIECE.search(c)
        square_first = m is not None
    if m:
        if square_first:
            color, piece, square = m.group(1), m.group(3).lower(), m.group(2).lower()
        else:
            color, piece, square = m.group(1), m.group(2).lower(), m.group(3).lower()
        sq = chess.parse_square(square)
        for name, b in (("before", board), ("after", after)):
            p = b.piece_at(sq)
            if p and p.piece_type == PIECE_TYPES[piece] and (not color or p.color == COLORS[color.lower()]):
                return _lib_res("correct", "piece_on_square", f"matches the board {name} the move")
        p = board.piece_at(sq)
        actual = "nothing" if p is None else f"a {'white' if p.color == chess.WHITE else 'black'} {chess.piece_name(p.piece_type)}"
        return _lib_res("incorrect", "piece_on_square", f"{chess.square_name(sq)} holds {actual} before the move, not a {piece}")

    if _CHECK.search(c):
        return _lib_res("correct" if board.gives_check(move) else "incorrect", "gives_check")

    nums = [float(x.replace("−", "-")) for x in _CPNUM.findall(c)]
    if nums and provided_evals:
        if _INTERPRET.search(c):
            return _lib_res("unknown", "echo_plus_interpretation")
        allowed = {float(v) for v in provided_evals}
        if all(n in allowed for n in nums):
            return _lib_res("correct", "echo_numbers", "numbers equal the evaluations given in the prompt")
        if all(n in allowed or -n in allowed for n in nums):
            return _lib_res("unknown", "echo_numbers", "sign differs from the White-perspective input")
        return _lib_res("incorrect", "echo_numbers", "number not in the prompt")

    if label in ("good", "bad") and classify(c)["category"] == "Quality evaluation" and re.search(r"\b(move|it|this|that)\b", c, re.I):
        pos, neg = bool(_POS.search(c)), bool(_NEGW.search(c))
        named = {x.lower() for x in re.findall(r"\b(white|black)(?:'s)?\b", c, re.I)}
        if named == {"black" if mover == chess.WHITE else "white"}:      # claim is about the opponent: polarity flips
            pos, neg = neg, pos
        if pos != neg:
            ok = (pos and label == "good") or (neg and label == "bad")
            return _lib_res("correct" if ok else "incorrect", "polarity_vs_label", f"claim {'positive' if pos else 'negative'}, Stockfish label {label}")
    return _lib_res("unknown", "no_check")


In [3]:
# Verbatim LLM claim splitting, validation and append-only caching.

SPLITTER_VERSION = "splitter-v1"
SYSTEM = "You split text into atomic statements. You never add, remove or change words."
USER_TEMPLATE = (
    "Split the chess explanation below into atomic claims.\n\n"
    "Rules:\n"
    "1. A claim is ONE statement that could be true or false on its own (one fact, one piece, one action).\n"
    "2. \"quote\" must be copied EXACTLY from the text, character for character. Never paraphrase, never fix grammar.\n"
    "3. If a quote lacks its subject because it was joined to an earlier part with \"and\" or a comma, put the subject words,\n"
    "   copied exactly from the text, in \"subject\". Otherwise leave \"subject\" empty.\n"
    "4. Keep the order of the text. Do not merge sentences. Do not skip any statement about the position or the move.\n"
    "5. Skip only pure filler that states nothing (for example \"In summary\").\n\n"
    "Example text: The knight attacks e5 and defends f3, which keeps the position solid.\n"
    "Example output: {{\"claims\": [{{\"quote\": \"The knight attacks e5\", \"subject\": \"\"}}, "
    "{{\"quote\": \"defends f3\", \"subject\": \"The knight\"}}, "
    "{{\"quote\": \"which keeps the position solid\", \"subject\": \"The knight\"}}]}}\n\n"
    "Text:\n\"\"\"\n{text}\n\"\"\"\n"
    "Return only JSON of the form {{\"claims\": [{{\"quote\": \"...\", \"subject\": \"...\"}}]}}."
)
SCHEMA = {"type": "object", "required": ["claims"], "properties": {"claims": {"type": "array", "items": {
    "type": "object", "required": ["quote", "subject"], "properties": {"quote": {"type": "string"}, "subject": {"type": "string"}}}}}}
MIN_COVERAGE = 0.80          # share of the explanation's letters/digits that the quotes must cover


def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[*_`#]+", "", s.replace("’", "'"))).strip().lower()


def alnum(s):
    return re.sub(r"[^a-z0-9]", "", norm(s))


def sha(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def settings_sha(settings):
    return sha(json.dumps([SPLITTER_VERSION, SYSTEM, USER_TEMPLATE, settings], sort_keys=True))


def ollama_chat(url, model, text, options, timeout=900):
    payload = dict(model=model, stream=False, keep_alive="15m", options=options, format=SCHEMA,
                   messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": USER_TEMPLATE.format(text=text)}])
    req = urllib.request.Request(url + "/api/chat", data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read()).get("message", {}).get("content", "")
    except urllib.error.URLError as e:
        raise RuntimeError(f"Cannot reach Ollama at {url}; start `ollama serve` and pull the splitter model.") from e


def validate(text, raw):
    """Return (claims, note). claims is None when the split must be rejected."""
    try:
        items = json.loads(raw)["claims"]
        assert isinstance(items, list) and items
    except Exception:
        return None, "invalid_json"
    t = norm(text)
    pos, out, covered = 0, [], 0
    for it in items:
        q, s = norm(str(it.get("quote", ""))), norm(str(it.get("subject", "")))
        if not q:
            continue
        i = t.find(q, pos)
        if i < 0:
            return None, f"quote_not_verbatim_or_out_of_order: {q[:60]}"
        if s and s not in t:
            return None, f"subject_not_verbatim: {s[:60]}"
        quote_sentences = [(si, norm(sent)) for si, sent in enumerate(split_sentences(text)) if q in norm(sent)]
        if not quote_sentences:
            return None, "quote_spans_sentence_boundary"
        if len(quote_sentences) > 1:
            return None, "quote_matches_multiple_sentences"
        sentence_idx, quote_sentence = quote_sentences[0]
        if s and s not in q and s not in quote_sentence:
            return None, f"subject_not_in_quote_sentence: {s[:60]}"
        pos = i + len(q)
        covered += len(alnum(q))
        out.append((i, q, s, str(it.get("quote", "")).strip(), str(it.get("subject", "")).strip(), sentence_idx))
    if not out:
        return None, "no_claims"
    if covered < MIN_COVERAGE * max(1, len(alnum(text))):
        return None, f"coverage_{covered / max(1, len(alnum(text))):.2f}"
    return out, ""


def to_claims(text, spans):
    sentences = split_sentences(text)
    sn = [norm(s) for s in sentences]
    out = []
    for k, (_, q, s, q_orig, s_orig, si) in enumerate(spans):
        claim = q_orig
        if norm(claim) not in norm(sentences[si]):
            raise ValueError(f"Splitter quote was not found in its associated source sentence: {claim[:80]}")
        out.append(dict(sentence_idx=si, clause_idx=k, sentence=sentences[si] if sentences else text, claim_text=claim,
                        source_quote=q_orig, source_subject=s_orig,
                        hedged=bool(_HEDGE.search(claim))))
    return out


def split_raw_result(text, raw):
    spans, note = validate(text, raw)
    if spans is None:
        return extract_claims(text), dict(split_source="rule_fallback", split_note=note)
    return to_claims(text, spans), dict(split_source="llm", split_note="")


def split_explanation(text, chat_fn):
    """chat_fn(text) -> raw JSON string. Returns claims and split metadata."""
    raw = chat_fn(text)
    claims, info = split_raw_result(text, raw)
    return claims, dict(info, raw=raw)


def split_all(texts, chat_fn, settings, cache_path):
    """texts: {key: text}. Cached by (text, settings). Returns {key: (claims, info)}; the cache file is JSONL and append-only."""
    cache_path = Path(cache_path)
    sset = settings_sha(settings)
    cache = {}
    if cache_path.exists():
        for line in cache_path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                d = json.loads(line)
                if d["settings_sha256"] == sset:
                    cache[d["text_sha256"]] = d
    results = {}
    print(f"Claim split progress: {len(texts)} explanations; {len(cache)} cached text(s) match the current splitter settings.", flush=True)
    with cache_path.open("a", encoding="utf-8", newline="\n") as f:
        for index, (key, text) in enumerate(texts.items(), start=1):
            h = sha(text)
            cache_hit = h in cache
            if not cache_hit:
                started = time.perf_counter()
                print(f"[{index}/{len(texts)}] Cache miss; requesting a splitter result.", flush=True)
                claims, info = split_explanation(text, chat_fn)
                cache[h] = dict(text_sha256=h, settings_sha256=sset, claims=claims, **info)
                f.write(json.dumps(cache[h], ensure_ascii=False) + "\n")
                f.flush()
                print(f"[{index}/{len(texts)}] Finished: {len(claims)} claim(s), {info['split_source']} ({time.perf_counter() - started:.1f}s).", flush=True)
            else:
                d = cache[h]
                if "raw" not in d:
                    raise ValueError(f"Cached split for {key} has no raw splitter response; set RUN_SPLITTER = True to rebuild it.")
                started = time.perf_counter()
                claims, info = split_raw_result(text, d["raw"])
                refreshed = dict(d, claims=claims, **info)
                if any(refreshed[name] != d.get(name) for name in ("claims", "split_source", "split_note")):
                    cache[h] = refreshed
                    f.write(json.dumps(refreshed, ensure_ascii=False) + "\n")
                    f.flush()
            d = cache[h]
            results[key] = (d["claims"], dict(split_source=d["split_source"], split_note=d["split_note"]))
            if cache_hit and (index == 1 or index % 5 == 0 or index == len(texts)):
                print(f"[{index}/{len(texts)}] Processed cached response: {d['split_source']}, {len(d['claims'])} claim(s), {time.perf_counter() - started:.2f}s for current item.", flush=True)
    return results


In [4]:
# Deterministic verification against chess positions and Stockfish evidence.

VERIFY_VERSION = "verify-v8.4"
VALUE = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}
_SAN = re.compile(r"\b(?:O-O-O|O-O|[KQRBN][a-h]?[1-8]?x?[a-h][1-8](?:=[QRBN])?|[a-h]x?[a-h][1-8](?:=[QRBN])?|[a-h][1-8](?:=[QRBN])?)[+#]?(?![a-z])")
_REAL_SAN = re.compile(r"\b(?:O-O-O|O-O|[KQRBN][a-h]?[1-8]?x?[a-h][1-8](?:=[QRBN])?[+#]?|[a-h]x[a-h][1-8](?:=[QRBN])?[+#]?)")
_PM = rf"(?:(?P<c>white|black)(?:'s)?\s+)?(?:the\s+)?(?:(?P<c2>white|black)(?:'s)?\s+)?(?P<p>{PIECE})s?(?:\s+(?:on|at|from)\s+(?P<s>{SQ}))?"
_REF = rf"(?:(?:the\s+)?(?:(?:white|black)(?:'s)?\s+)?{PIECE}s?(?:\s+(?:on|at)\s+{SQ})?|{SQ})"
_SUBJ_SAN = r"(?:[KQRBN][a-h]?[1-8]?x?[a-h][1-8]|[a-h]x[a-h][1-8]|[a-h][1-8])[+#]?"
_MOVE_CHOICE = re.compile(rf"\b(?:choose|chooses|chose|chosen|play|plays|played|playing|select|selects|selected|selecting|make|makes|made)\s+(?:(?:the\s+)?move\s+)?(?P<san>{_SUBJ_SAN})(?![a-z])", re.I)
_PAWN_ADVANCE = re.compile(rf"\b(?:the\s+)?move\s+(?P<san>{_SUBJ_SAN})\s+advanc(?:e|es|ed|ing)\s+(?:the\s+)?pawn\b", re.I)
_ATK = re.compile(rf"(?P<subj>{_PM}|\b{_SUBJ_SAN}(?=\s)|\bit\b|\bthis move\b|\bthe move\b)\s+(?:now\s+|also\s+|directly\s+)?(?P<v>attacks?|threatens?|targets?|eyes|hits|pressur(?:es|ing))\s+(?P<obj>{_REF})", re.I)
_DEF = re.compile(rf"(?P<subj>{_PM}|\b{_SUBJ_SAN}(?=\s)|\bit\b|\bthis move\b|\bthe move\b)\s+(?:now\s+|also\s+)?(?P<v>defends?|protects?|guards?|supports?)\s+(?P<obj>{_REF})", re.I)
_FRAMING = re.compile(r"^(?:a|the)\s+(?:human|player|human player)\s+(?:might|may|could|would|can)\b|\b(?:human|player)\s+(?:might|may|could|would)\s+(?:choose|play|make|prefer|select|opt)\b|"
                      r"^(?:in summary|overall|in conclusion|to summari[sz]e)\b", re.I)
_BEST = re.compile(r"\b(?:the\s+)?(?:best|top|optimal|strongest|most accurate|engine'?s?(?:\s+(?:first|top))?)\s+(?:move|choice|option|continuation|recommendation)\b|\bengine'?s?\s+(?:top\s+)?(?:choice|recommend\w*)\b", re.I)
_MOVE_SUBJ = re.compile(r"\b(?:this|the|it|move|played)\b", re.I)
_LEVEL = re.compile(r"\b(?:equal|balanced|level|equali[sz]e\w*|drawish)\b", re.I)
_BETTER = re.compile(r"\b(?:better|advantage|ahead|favou?red|edge|upper hand|plus)\b", re.I)
_WINNING = re.compile(r"\b(?:winning|decisive|crushing|dominant)\b", re.I)
_LOSING = re.compile(r"\b(?:losing|lost|worse|disadvantage)\b", re.I)
_BLUNDER = re.compile(r"\b(blunder\w*|mistake\w*|inaccura\w*)\b", re.I)
_WINS = re.compile(rf"\b(?:wins?|winning|gains?|gaining|picks? up|nets?)\s+(?:a|an|the|some)?\s*(?:extra\s+|free\s+)?(?:(pawn)|(knight|bishop|minor piece)|(rook|exchange)|(queen)|(material))\b", re.I)
_BETTER_ALT = re.compile(r"\b(?:better|stronger|preferable|superior|more accurate|instead|should have|rather than|best alternative|would have been)\b", re.I)
_ALSO_OK = re.compile(r"\b(?:also|equally)\s+(?:good|strong|viable|playable|possible|reasonable)\b|\b(?:viable|playable)\b", re.I)


def verify_res(verdict, rule, detail="", source="python_chess", cat=None):
    return dict(auto_verdict=verdict, auto_rule=rule, auto_detail=detail, auto_source=source, check_category=cat)


NONE = dict(auto_verdict="unknown", auto_rule="no_check", auto_detail="", auto_source="none", check_category=None)
CAT_OF_V1 = {"capture_identity": "Tactical reasoning", "gives_check": "Tactical reasoning", "moved_piece": "Board comprehension",
             "piece_on_square": "Board comprehension", "echo_numbers": "Quality evaluation", "polarity_vs_label": "Quality evaluation"}


def is_framing(claim):
    return bool(_FRAMING.search(claim.strip()))


def _piece_filter(board, color, ptype, sq):
    """pieces of `color` (None = any) matching type/square."""
    out = []
    for s, p in board.piece_map().items():
        if (color is None or p.color == color) and (ptype is None or p.piece_type == ptype) and (sq is None or s == sq):
            out.append(s)
    return out


def _parse_ref(text, default_color=None):
    """('piece'|'square', color|None, ptype|None, square|None)"""
    m = re.search(rf"(?:(white|black)(?:'s)?\s+)?{PIECE}s?(?:\s+(?:on|at)\s+({SQ}))?", text, re.I)
    if m:
        return (COLOR(m.group(1)), PIECE_TYPES[m.group(2).lower()], chess.parse_square(m.group(3).lower()) if m.group(3) else None)
    m = re.search(SQ, text, re.I)
    if m:
        return (None, None, chess.parse_square(m.group(0).lower()))
    return None


def COLOR(name):
    return COLORS[name.lower()] if name else None


def _subject(board, after, move, subj, mover):
    """squares (in `after`) of the pieces the subject can refer to, plus their colour."""
    s = subj.strip().lower()
    if s in ("it", "this move", "the move") or re.fullmatch(_SUBJ_SAN, subj.strip()):
        if s not in ("it", "this move", "the move") and subj.strip().rstrip("+#") != board.san(move).rstrip("+#"):
            return [], mover            # a SAN that is not the played move: cannot resolve here
        return [move.to_square], mover
    m = re.search(rf"(?:(white|black)(?:'s)?\s+)?(?:the\s+)?(?:(white|black)(?:'s)?\s+)?{PIECE}s?(?:\s+(?:on|at|from)\s+({SQ}))?", s, re.I)
    if not m:
        return [], mover
    color = COLOR(m.group(1) or m.group(2)) if (m.group(1) or m.group(2)) else mover
    ptype = PIECE_TYPES[m.group(3).lower()]
    sq = chess.parse_square(m.group(4).lower()) if m.group(4) else None
    sqs = _piece_filter(after, color, ptype, sq)
    moved = [x for x in sqs if x == move.to_square]
    return (moved or sqs), color


def _tactic(c, board, after, move, mover):
    for rx, kind in ((_ATK, "attacks"), (_DEF, "defends")):
        m = rx.search(c)
        if not m or _MODAL.search(c[:m.start()][-30:]):
            continue
        src, color = _subject(board, after, move, m.group("subj"), mover)
        ref = _parse_ref(m.group("obj"))
        if not src or not ref:
            return verify_res("unknown", f"{kind}_unresolved", cat="Tactical reasoning")
        tcol, tptype, tsq = ref
        hit = False
        for a in src:
            for t in after.attacks(a):
                p = after.piece_at(t)
                if p is None:
                    ok = tptype is None and tsq == t
                else:
                    side_ok = (p.color != color) if kind == "attacks" else (p.color == color and t != a)
                    ok = side_ok and (tptype is None or p.piece_type == tptype) and (tsq is None or t == tsq) and (tcol is None or p.color == tcol)
                if ok:
                    hit = True
        if kind == "attacks" and tsq is not None and tptype is None and not hit:
            hit = any(tsq in after.attacks(a) for a in src)
        return verify_res("correct" if hit else "incorrect", kind, f"checked on the board after the move", cat="Tactical reasoning")
    return None


def _material(board, move, pv_after, mover):
    """material change (mover's view) from the played move plus the stored PV; None when the PV ends in the middle of an exchange."""
    b = board.copy()
    gain = 0
    if b.is_capture(move):
        victim = chess.Piece(chess.PAWN, not mover) if b.is_en_passant(move) else b.piece_at(move.to_square)
        gain += VALUE[victim.piece_type]
    b.push(move)
    last_cap = b.is_capture(move) if False else False
    for san in pv_after:
        try:
            m = b.parse_san(san)
        except ValueError:
            return None
        cap = b.is_capture(m)
        if cap:
            v = chess.Piece(chess.PAWN, not b.turn) if b.is_en_passant(m) else b.piece_at(m.to_square)
            gain += VALUE[v.piece_type] * (1 if b.turn == mover else -1)
        last_cap = cap
        b.push(m)
    return None if last_cap or len(pv_after) < 4 else gain


_TOK = r"(?:O-O-O|O-O|[KQRBN][a-h]?[1-8]?x?[a-h][1-8](?:=[QRBN])?|[a-h]x[a-h][1-8](?:=[QRBN])?|[a-h][1-8](?:=[QRBN])?)[+#]?"
_NUM = r"(?:\d+\.(?:\.\.)?\s*)?"
_RUN = re.compile(rf"(?<![\w]){_NUM}{_TOK}(?:\s*(?:,|->|→|\s)\s*{_NUM}{_TOK})+(?![\w])")


def _san_tokens(c):
    """single-move mentions: piece moves, captures, castling, plus a bare pawn push when it is clearly a move ('d4 would be better')."""
    toks = [t.rstrip("+#") for t in _REAL_SAN.findall(c)]
    for m in re.finditer(r"(?<![\w])([a-h][1-8])(?=\s+(?:would|is\s+(?:better|stronger|preferable|also|a)|instead|should)\b)", c):
        if not re.search(r"\b(?:on|at|to|from|the|of)\s*$", c[:m.start()]):
            toks.append(m.group(1))
    return toks


def _seq_tokens(c):
    """a run of >= 2 consecutive move tokens (pawn pushes allowed inside the run, at least one non-bare-square token required)."""
    for m in _RUN.finditer(c):
        toks = [t.rstrip("+#") for t in re.findall(_TOK, re.sub(_NUM, "", m.group(0)))]
        if any(not re.fullmatch(r"[a-h][1-8]", t) for t in toks):
            return toks
    return []


def _stockfish_stored(c, board, after, move, mover, ctx):
    label, cp_loss = ctx.get("label"), ctx.get("cp_loss")
    side_names = {x.lower() for x in re.findall(r"\b(white|black)\b", c, re.I)}
    named_color = None if len(side_names) != 1 else COLORS[next(iter(side_names))]

    # "best move / engine's choice": about the played move, or names a different best move
    m = _BEST.search(c)
    if m and not re.search(r"\b(?:one of|among|a)\s+(?:the\s+)?(?:best|top)\b|\bnot\b", c, re.I):
        toks = _san_tokens(c)
        if toks and ctx.get("best_san"):
            ok = ctx["best_san"].rstrip("+#") in toks and ctx["best_san"].rstrip("+#") != board.san(move).rstrip("+#") or ctx.get("human_is_best")
            if len(toks) == 1:
                same = toks[0] == ctx["best_san"].rstrip("+#")
                return verify_res("correct" if same else "incorrect", "best_move_named", f"engine best move: {ctx['best_san']}", "stockfish_stored", "Quality evaluation")
        if ctx.get("human_is_best") is not None and _MOVE_SUBJ.search(c[:m.start() + 20]) and not toks:
            return verify_res("correct" if ctx["human_is_best"] else "incorrect", "best_move_played",
                        "played move equals the engine's best move" if ctx["human_is_best"] else "engine prefers another move", "stockfish_stored", "Quality evaluation")

    # blunder / mistake / inaccuracy: cp-loss bands (Lichess-like; stated in the manifest)
    m = _BLUNDER.search(c)
    if m and cp_loss is not None and re.search(r"\b(move|it|this|that|play)\b", c, re.I):
        w = m.group(1).lower()
        lo, hi = (200, 10**9) if w.startswith("blunder") else (100, 200) if w.startswith("mistake") else (50, 100)
        return verify_res("correct" if lo <= cp_loss < hi else "incorrect", "error_band", f"cp loss {cp_loss}, band for '{w}' = [{lo},{hi})", "stockfish_stored", "Quality evaluation")

    # who stands better after the move (named side only)
    if named_color is not None and ctx.get("eval_after_white") is not None and re.search(r"\b(?:is|are|stands?|has|have|gets?|remains?|keeps?|holds?)\b", c, re.I):
        e = ctx["eval_after_white"] * (1 if named_color == chess.WHITE else -1)
        if _LEVEL.search(c):
            return verify_res("correct" if abs(e) <= 60 else "incorrect", "eval_equal", f"eval after the move {ctx['eval_after_white']:+d} cp (White)", "stockfish_stored", "Quality evaluation")
        if _WINNING.search(c):
            return verify_res("correct" if e >= 300 else ("incorrect" if e <= 100 else "unknown"), "eval_winning", f"{e:+d} cp for the named side", "stockfish_stored", "Quality evaluation")
        if _LOSING.search(c):
            return verify_res("correct" if e <= -150 else ("incorrect" if e >= -40 else "unknown"), "eval_losing", f"{e:+d} cp for the named side", "stockfish_stored", "Quality evaluation")
        if _BETTER.search(c):
            return verify_res("correct" if e >= 40 else ("incorrect" if e <= -40 else "unknown"), "eval_better", f"{e:+d} cp for the named side", "stockfish_stored", "Quality evaluation")

    # wins material (played capture + stored PV after the move)
    m = _WINS.search(c)
    if m and not _MODAL.search(c[:m.start()][-30:]) and ctx.get("pv_after"):
        want = 1 if m.group(1) else 3 if m.group(2) else 2 if m.group(3) else 9 if m.group(4) else 1
        g = _material(board, move, ctx["pv_after"], mover)
        if g is not None:
            return verify_res("correct" if g >= want else "incorrect", "pv_material", f"material change for the mover along the engine line: {g:+d}", "stockfish_stored", "Tactical reasoning")

    # sequences of moves: legal? equal to the engine line?
    toks = _seq_tokens(c)
    if len(toks) >= 2:
        best = [t.rstrip("+#") for t in ctx.get("best_pv", [])]
        if best and any(best[i:i + len(toks)] == toks for i in range(len(best))):
            return verify_res("correct", "pv_echo", "sequence occurs in the engine's principal variation", "stockfish_stored", "Tactical reasoning")
        for start in (board, after):
            b = start.copy()
            try:
                for t in toks:
                    b.push_san(t)
                return verify_res("unknown", "sequence_legal_not_in_pv", "legal sequence that is not the engine line")
            except ValueError:
                continue
        return verify_res("incorrect", "sequence_illegal", "the named move sequence is not legal from this position", "python_chess", "Legality")
    return None


def _live(c, board, move, mover, ctx, engine):
    toks = _san_tokens(c)
    if len(toks) != 1 or not _BETTER_ALT.search(c) and not _ALSO_OK.search(c):
        return None
    try:
        alt = board.parse_san(toks[0])
    except ValueError:
        if _BETTER_ALT.search(c):
            return verify_res("incorrect", "named_move_illegal", f"{toks[0]} is not a legal move here", "python_chess", "Legality")
        return None
    if alt == move:
        return None
    if engine is None:
        return verify_res("unknown", "live_engine_off", "alternative move named; run with RUN_LIVE_STOCKFISH", "none", "Quality evaluation")
    alt_e = engine.score_after(board.fen(), alt.uci())
    played_e = ctx["eval_after_mover"]
    if _BETTER_ALT.search(c):
        d = alt_e - played_e
        return verify_res("correct" if d >= 30 else ("incorrect" if d <= 0 else "unknown"), "alt_better", f"{toks[0]} {alt_e:+d} cp vs played {played_e:+d} cp (mover view)", "stockfish_live", "Quality evaluation")
    loss = ctx["eval_before_mover"] - alt_e
    return verify_res("correct" if loss <= 30 else ("incorrect" if loss >= 100 else "unknown"), "alt_also_ok", f"{toks[0]} loses {loss} cp vs best", "stockfish_live", "Quality evaluation")


def _threat_san(c, after, mover):
    """'threatens Bxf7': would that capture be legal if the mover could move again (null move)?"""
    m = re.search(r"\b(?:threat\w*|threatens?|intends?)\s+(?:to\s+)?(?:play\s+)?(" + _REAL_SAN.pattern + ")", c)
    if not m or after.is_check():
        return None
    b = after.copy(); b.push(chess.Move.null())
    try:
        mv = b.parse_san(m.group(1).rstrip("+#"))
    except ValueError:
        return verify_res("incorrect", "threat_illegal", f"{m.group(1)} would not be legal for the mover", cat="Tactical reasoning")
    return verify_res("correct", "threat_san", "the named move would be legal after a passing move", cat="Tactical reasoning")


def verify(claim, board, move, ctx, engine=None):
    c = claim.replace("’", "'")
    if _NEG.search(c):
        return verify_res("unknown", "negation_unsupported")
    mover = board.turn
    after = board.copy(); after.push(move)
    moved = board.piece_at(move.from_square)
    advance = _PAWN_ADVANCE.search(c)
    if advance and not _MODAL.search(c[:advance.start()][-30:]):
        san = advance.group("san")
        try:
            named_move = board.parse_san(san)
        except ValueError:
            return verify_res("incorrect", "played_move_named", f"{san} is not legal from this position", cat="Board comprehension")
        if named_move != move:
            return verify_res("incorrect", "played_move_named", f"played move is {board.san(move)}; claim names {san}", cat="Board comprehension")
        if moved.piece_type != chess.PAWN:
            return verify_res("incorrect", "pawn_advance", f"played move moves a {chess.piece_name(moved.piece_type)}, not a pawn", cat="Board comprehension")
        forward = 1 if mover == chess.WHITE else -1
        if (chess.square_rank(move.to_square) - chess.square_rank(move.from_square)) * forward <= 0:
            return verify_res("incorrect", "pawn_advance", "played pawn move is not a forward advance", cat="Board comprehension")
        return verify_res("correct", "pawn_advance", f"pawn advances from {chess.square_name(move.from_square)} to {chess.square_name(move.to_square)}", cat="Board comprehension")
    chosen = _MOVE_CHOICE.search(c)
    if chosen and not _MODAL.search(c[:chosen.start()][-30:]):
        san = chosen.group("san")
        try:
            named_move = board.parse_san(san)
        except ValueError:
            return verify_res("unknown", "played_move_unparsed", f"could not resolve {san} in this position", "none", "Board comprehension")
        actual_san = board.san(move)
        return verify_res("correct" if named_move == move else "incorrect", "played_move_named",
                          f"played move is {actual_san}; claim names {san}", cat="Board comprehension")

    # 1. relational/tactical claims first: v1's piece_on_square would otherwise accept 'the bishop attacks the knight on c6' just because a knight is on c6
    if re.search(r"\bcastl\w+", c, re.I) and not _MODAL.search(c[:re.search(r"\bcastl", c, re.I).start()][-30:]):
        return verify_res("correct" if board.is_castling(move) else "incorrect", "castles", cat="Board comprehension")
    mm = re.search(r"\b(?:check\s?mate|mates?|delivers? mate)\b", c, re.I)
    if mm and not _MODAL.search(c[:mm.start()][-30:]):
        return verify_res("correct" if after.is_checkmate() else "incorrect", "checkmate", cat="Tactical reasoning")
    t = _tactic(c, board, after, move, mover) or _threat_san(c, after, mover)
    if t:
        return t
    m = re.search(rf"\bdevelop\w*\s+(?:the\s+)?{PIECE}\b", c, re.I)
    if m and not _MODAL.search(c[:m.start()][-30:]):
        pt = PIECE_TYPES[m.group(1).lower()]
        home = chess.square_rank(move.from_square) == (0 if mover == chess.WHITE else 7)
        return verify_res("correct" if (moved.piece_type == pt and home) else "incorrect", "develops",
                    f"moved piece: {chess.piece_name(moved.piece_type)} from {chess.square_name(move.from_square)}", cat="Board comprehension")
    m = re.search(r"\bfork\w*\b", c, re.I)
    if m and not _MODAL.search(c[:m.start()][-30:]):
        n = sum(1 for t in after.attacks(move.to_square) if (p := after.piece_at(t)) and p.color != mover)
        return verify_res("correct" if n >= 2 else "incorrect", "fork", f"moved piece attacks {n} enemy piece(s)", cat="Tactical reasoning")
    m = re.search(rf"\bpin\w*\b.*?{PIECE}|{PIECE}\b.*?\bpinned\b", c, re.I)
    if m:
        ref = _parse_ref(c[m.start():])
        if ref:
            col, pt, sq = ref
            sqs = _piece_filter(after, col if col is not None else (not mover), pt, sq)
            if any(after.is_pinned(after.piece_at(s).color, s) for s in sqs):
                return verify_res("correct", "pin", "absolute pin on the board after the move", cat="Tactical reasoning")
            return verify_res("unknown", "pin_not_absolute", "no absolute pin found (a relative pin is possible)", "none", "Tactical reasoning")
    m = re.search(rf"{_PM}\s+(?:is|are|remains?)\s+(?:left\s+)?(?P<w>undefended|unprotected|hanging|en prise)", c, re.I)
    if m and m.group("s"):
        sq = chess.parse_square(m.group("s").lower())
        p = after.piece_at(sq)
        if p is None or p.piece_type != PIECE_TYPES[m.group("p").lower()]:
            return verify_res("incorrect", "hanging_piece_missing", f"no such piece on {chess.square_name(sq)}", cat="Board comprehension")
        defended = after.is_attacked_by(p.color, sq)
        attacked = after.is_attacked_by(not p.color, sq)
        if m.group("w").lower() in ("undefended", "unprotected"):
            return verify_res("correct" if not defended else "incorrect", "undefended", cat="Tactical reasoning")
        return verify_res("correct" if (attacked and not defended) else ("incorrect" if not attacked else "unknown"), "hanging", cat="Tactical reasoning")

    # 2. stored Stockfish results (best move, error bands, who is better, material along the PV, move sequences)
    s = _stockfish_stored(c, board, after, move, mover, ctx)
    if s:
        return s
    # 3. live Stockfish for alternative moves named in the explanation
    if engine is not None or re.search(r"\b(?:better|stronger|preferable|instead|should have|viable|playable|alternative)\b", c, re.I):
        lv = _live(c, board, move, mover, ctx, engine)
        if lv:
            return lv
    # 4. v1 checks: capture identity, moved piece, piece on square, check, echoed numbers, polarity vs label
    v1 = precheck(c, board, move, ctx.get("label"), tuple(ctx.get("provided_evals", ())))
    if v1["auto_verdict"] != "unknown":
        src = "label" if v1["auto_rule"] == "polarity_vs_label" else "python_chess"
        return verify_res(v1["auto_verdict"], v1["auto_rule"], v1["auto_detail"], src, CAT_OF_V1.get(v1["auto_rule"]))
    return dict(NONE)


# ------------------------------------------------------------------ live Stockfish (optional; same settings as Step 2)
class LiveEngine:
    """Scores the position after a given move from the mover's perspective. Cached on disk so reruns are identical and need no engine."""

    def __init__(self, path, depth, threads, hash_mb, cache_path=None):
        import json
        from pathlib import Path
        self.depth, self.cache_path, self.cache = depth, Path(cache_path) if cache_path else None, {}
        self.key_prefix = f"d{depth}t{threads}h{hash_mb}"
        if self.cache_path and self.cache_path.exists():
            self.cache = {k: v for k, v in (json.loads(l) for l in self.cache_path.read_text(encoding="utf-8").splitlines() if l.strip())}
        self.engine = chess.engine.SimpleEngine.popen_uci(str(path))
        self.engine.configure({"Threads": threads, "Hash": hash_mb})
        self.version = self.engine.id.get("name")

    def score_after(self, fen, uci):
        import json
        key = f"{self.key_prefix}|{fen}|{uci}"
        if key not in self.cache:
            b = chess.Board(fen); mover = b.turn; b.push_uci(uci)
            info = self.engine.analyse(b, chess.engine.Limit(depth=self.depth))
            cp = info["score"].pov(mover).score(mate_score=10000)
            self.cache[key] = int(max(-10000, min(10000, cp)))
            if self.cache_path:
                with self.cache_path.open("a", encoding="utf-8", newline="\n") as f:
                    f.write(json.dumps([key, self.cache[key]]) + "\n")
        return self.cache[key]

    def close(self):
        self.engine.quit()


# ------------------------------------------------------------------ the 'Verdict:' line of the v2 prompt
VERDICT_RE = re.compile(r"(?im)^[ \t>*_#-]*verdict[ \t*_]*:[ \t*_]*(good|inaccurate|bad)\b[^\n]*$")


def split_verdict(text):
    """-> (explanation_text_without_verdict_line, verdict or None). Strips a leading 'Explanation:' label."""
    m = None
    for m in VERDICT_RE.finditer(text):
        pass
    verdict = m.group(1).lower() if m else None
    body = (text[:m.start()] + text[m.end():]) if m else text
    body = re.sub(r"(?im)^[ \t*_#-]*explanation[ \t*_]*:[ \t*_]*", "", body).strip()
    return body, verdict


def verdict_check(verdict, label):
    """good <-> Stockfish label 'good' (cp loss <= 30); inaccurate/bad <-> label 'bad' (cp loss >= 100)."""
    ok = (verdict == "good") == (label == "good")
    return verify_res("correct" if ok else "incorrect", "verdict_vs_label", f"verdict {verdict}, Stockfish label {label}", "label", "Quality evaluation")


def _strict_subject_squares(board_before, board_after, move, subject, mover):
    s = str(subject or "").strip()
    if not s:
        return [], None, "missing_subject"
    if s.lower() in {"it", "this move", "the move"}:
        p = board_after.piece_at(move.to_square)
        return ([move.to_square], p.color, "") if p else ([], None, "moved_piece_missing")
    if re.fullmatch(_SUBJ_SAN, s):
        if s.rstrip("+#") == board_before.san(move).rstrip("+#"):
            p = board_after.piece_at(move.to_square)
            return ([move.to_square], p.color, "") if p else ([], None, "moved_piece_missing")
        return [], None, "subject_san_not_played_move"

    m = re.search(
        rf"(?:(?P<color1>white|black)(?:'s)?\s+)?(?:the\s+)?(?:(?P<color2>white|black)(?:'s)?\s+)?"
        rf"(?P<piece>king|queen|rook|bishop|knight|pawn)s?(?:\s+(?:on|at|from)\s+(?P<square>{SQ}))?",
        s, re.I
    )
    if not m:
        return [], None, "subject_not_resolved"
    colors = [x.lower() for x in (m.group("color1"), m.group("color2")) if x]
    if len(set(colors)) > 1:
        return [], None, "conflicting_subject_colors"
    color = COLORS[colors[0]] if colors else mover
    piece_type = PIECE_TYPES[m.group("piece").lower()]
    square = chess.parse_square(m.group("square").lower()) if m.group("square") else None
    candidates = [
        sq for sq, piece in board_after.piece_map().items()
        if piece.color == color and piece.piece_type == piece_type and (square is None or sq == square)
    ]
    if len(candidates) != 1:
        if square is not None and square == move.from_square:
            moved_before = board_before.piece_at(move.from_square)
            moved_after = board_after.piece_at(move.to_square)
            if (moved_before is not None and moved_after is not None
                    and moved_before.piece_type == piece_type
                    and moved_before.color == color
                    and moved_after.piece_type == piece_type
                    and moved_after.color == color):
                return [move.to_square], moved_after.color, ""
        return [], color, "ambiguous_subject" if candidates else "subject_piece_not_found"
    return candidates, color, ""


def _strict_relation(claim, board, after, move, mover, source_subject=""):
    if re.search(r"\b(?:threatens?|threatening|threat)\b", claim, re.I):
        return verify_res("unknown", "threat_requires_variation",
                          "a legal-looking move alone does not establish a real threat",
                          "none", "Tactical reasoning")

    rx = re.compile(
        rf"(?P<subj>{_PM}|\b{_SUBJ_SAN}(?=\s)|\bit\b|\bthis move\b|\bthe move\b)"
        rf"\s+(?:now\s+|also\s+|directly\s+)?"
        rf"(?P<verb>attacks?|targets?|eyes|hits|pressur(?:es|ing)|defends?|protects?|guards?|supports?)"
        rf"\s+(?P<obj>{_REF})", re.I
    )
    m = rx.search(claim)
    if not m:
        return None

    subject = str(source_subject or m.group("subj")).strip()
    subject_check = precheck(subject, board, move)
    if subject_check["auto_verdict"] == "incorrect":
        return verify_res(subject_check["auto_verdict"], subject_check["auto_rule"], subject_check["auto_detail"],
                          "python_chess", CAT_OF_V1.get(subject_check["auto_rule"]))
    sources, _, reason = _strict_subject_squares(board, after, move, subject, mover)
    if not sources:
        return verify_res("unknown", "relation_subject_unresolved", reason, "none", "Tactical reasoning")

    square_first = _SQUARE_PIECE.search(m.group("obj"))
    if square_first:
        target_color = COLORS[square_first.group(1).lower()] if square_first.group(1) else None
        target_square = chess.parse_square(square_first.group(2).lower())
        target_type = PIECE_TYPES[square_first.group(3).lower()]
    else:
        target = _parse_ref(m.group("obj"))
        if target is None:
            return verify_res("unknown", "relation_target_unresolved", "target could not be resolved", "none", "Tactical reasoning")
        target_color, target_type, target_square = target
    if target_square is None and target_type is not None:
        candidates = [
            sq for sq, piece in after.piece_map().items()
            if piece.piece_type == target_type and (target_color is None or piece.color == target_color)
        ]
        if len(candidates) != 1:
            return verify_res("unknown", "relation_target_ambiguous", "target piece is not uniquely identified",
                              "none", "Tactical reasoning")
        target_square = candidates[0]
    if target_square is None:
        return verify_res("unknown", "relation_target_unresolved", "no target square identified", "none", "Tactical reasoning")

    target_piece = after.piece_at(target_square)
    if target_type is not None and (target_piece is None or target_piece.piece_type != target_type):
        return verify_res("incorrect", "relation_target_piece", f"{chess.square_name(target_square)} does not hold the named piece",
                          "python_chess", "Tactical reasoning")
    if target_color is not None and (target_piece is None or target_piece.color != target_color):
        return verify_res("incorrect", "relation_target_color", f"target on {chess.square_name(target_square)} is not that color",
                          "python_chess", "Tactical reasoning")

    source_square = sources[0]
    source_piece = after.piece_at(source_square)
    if source_piece is None:
        return verify_res("unknown", "relation_source_missing", "source piece is absent", "none", "Tactical reasoning")
    verb = m.group("verb").lower()
    kind = "defends" if verb.startswith(("defend", "protect", "guard", "support")) else "attacks"
    if kind == "attacks":
        if target_piece is not None and target_piece.color == source_piece.color:
            return verify_res("incorrect", "attacks_friendly_piece", "the target is on the same side as the source piece",
                              "python_chess", "Tactical reasoning")
    elif target_piece is None:
        return verify_res("unknown", "defends_empty_square", "a defence claim needs an identifiable friendly piece",
                          "none", "Tactical reasoning")
    elif target_piece.color != source_piece.color:
        return verify_res("incorrect", "defends_enemy_piece", "the target is not a friendly piece",
                          "python_chess", "Tactical reasoning")
    hit = target_square in after.attacks(source_square)
    detail = f"{chess.square_name(source_square)} geometrically {kind} {chess.square_name(target_square)} after the move"
    return verify_res("correct" if hit else "incorrect", kind, detail, "python_chess", "Tactical reasoning")


def _explicit_square_claim(claim, board, after):
    loc = re.fullmatch(
        rf"\s*(?:(?P<when>before|after)(?:\s+the)?\s+move\s*[:,]?\s*)?"
        rf"(?:the\s+)?(?:(?P<color>white|black)(?:'s)?\s+)?"
        rf"(?P<piece>king|queen|rook|bishop|knight|pawn)s?\s+"
        rf"(?:is|are|sits|stands|remains|is\s+located|is\s+positioned|is\s+placed)\s+"
        rf"(?:currently\s+|now\s+)?on\s+(?P<square>{SQ})\s*[.!?]*\s*", claim, re.I
    )
    if not loc:
        return None
    color = COLORS[loc.group("color").lower()] if loc.group("color") else None
    piece_type = PIECE_TYPES[loc.group("piece").lower()]
    square_name = loc.group("square").lower()
    square = chess.parse_square(square_name)
    position = board if loc.group("when") == "before" else after
    actual = position.piece_at(square)
    correct = bool(actual and actual.piece_type == piece_type and (color is None or actual.color == color))
    if correct:
        return verify_res("correct", "piece_on_square", f"{square_name} contains the claimed piece in that position",
                          "python_chess", "Board comprehension")
    actual_name = "nothing" if actual is None else f"a {'white' if actual.color == chess.WHITE else 'black'} {chess.piece_name(actual.piece_type)}"
    when = "before the move" if loc.group("when") == "before" else "after the move"
    return verify_res("incorrect", "piece_on_square", f"{square_name} contains {actual_name} in the {when} position",
                      "python_chess", "Board comprehension")


_LEADING_CLAIM_MARKER = re.compile(
    r"^(?:and|but|or|so|yet|also|furthermore|moreover|additionally|however|thus|therefore|then|plus)\b[,;:]?\s+", re.I
)
_SQUARE_FIRST_PIECE = re.compile(
    r"\b(the\s+(?:white\s+|black\s+)?)([a-h][1-8])-(king|queen|rook|bishop|knight|pawn)s?\b", re.I
)


def normalize_claim_for_check(text):
    normalized = str(text or "").strip()
    while True:
        normalized, count = _LEADING_CLAIM_MARKER.subn("", normalized, count=1)
        if not count:
            break
        normalized = normalized.strip()
    return _SQUARE_FIRST_PIECE.sub(
        lambda match: f"{match.group(1)}{match.group(3)} on {match.group(2)}", normalized
    )


def verify_claim(claim_item, board, move, ctx, engine=None):
    claim_text = normalize_claim_for_check(claim_item.get("claim_text"))
    subject = str(claim_item.get("source_subject") or "").strip()
    full_claim = f"{subject} {claim_text}".strip() if subject and norm(subject) not in norm(claim_text) else claim_text
    full_claim = normalize_claim_for_check(full_claim)
    if _NEG.search(full_claim):
        return verify_res("unknown", "negation_requires_review", "negated claim needs manual interpretation",
                          "none", classify(full_claim)["category"])
    if subject:
        check = precheck(subject, board, move)
        if check["auto_verdict"] == "incorrect":
            return verify_res(check["auto_verdict"], check["auto_rule"], check["auto_detail"],
                              "python_chess", CAT_OF_V1.get(check["auto_rule"]))
    if _CAPTURE_ACTOR.search(full_claim):
        check = precheck(full_claim, board, move, ctx.get("label"), tuple(ctx.get("provided_evals", ())))
        if check["auto_verdict"] == "incorrect":
            return verify_res(check["auto_verdict"], check["auto_rule"], check["auto_detail"],
                              "python_chess", CAT_OF_V1.get(check["auto_rule"]))

    after = board.copy()
    after.push(move)
    mover = board.turn
    location = _explicit_square_claim(full_claim, board, after)
    if location is not None:
        return location
    relation = _strict_relation(full_claim, board, after, move, mover, subject)
    if relation is not None:
        return relation
    if re.search(r"\bfork\w*\b", full_claim, re.I):
        sources, _, reason = _strict_subject_squares(board, after, move, subject or "this move", mover)
        if not sources:
            return verify_res("unknown", "fork_subject_unresolved", reason, "none", "Tactical reasoning")
        targets = [after.piece_at(sq) for sq in after.attacks(sources[0])]
        enemy = [piece for piece in targets if piece is not None and piece.color != mover]
        major = [piece for piece in enemy if piece.piece_type in (chess.KING, chess.QUEEN, chess.ROOK)]
        if len(enemy) < 2:
            return verify_res("incorrect", "fork_targets", f"source attacks only {len(enemy)} enemy piece(s)",
                              "python_chess", "Tactical reasoning")
        if major:
            return verify_res("correct", "fork_targets", "source attacks multiple enemy pieces including a major piece",
                              "python_chess", "Tactical reasoning")
        return verify_res("unknown", "fork_minor_targets_only", "multiple targets do not include a king, queen, or rook",
                          "none", "Tactical reasoning")

    result = verify(full_claim, board, move, ctx, engine)
    if result.get("auto_rule") in {"piece_on_square", "threat_san"}:
        return verify_res("unknown", "claim_needs_context", "legacy check did not establish the full claim",
                          "none", result.get("check_category"))
    return result


def run_verifier_self_tests():
    assert is_framing("Overall")
    assert not is_framing("Pawn advances")
    ctx = dict(label="bad", cp_loss=148, eval_before_mover=73, eval_after_mover=-75,
               eval_after_white=75, best_san="c5", best_pv=["c5"], pv_after=["Kxg2"],
               human_is_best=False, provided_evals=())
    b = chess.Board("4k3/8/2n5/8/4p3/8/8/4K2B w - - 0 1")
    move = b.parse_san("Bxe4")
    assert verify_claim({"claim_text": "attacks the knight on c6", "source_subject": "The bishop"}, b, move, ctx)["auto_verdict"] == "correct"
    normalized_claim = {"claim_text": "and attacks the c6-knight", "source_subject": "The bishop"}
    assert verify_claim(normalized_claim, b, move, ctx)["auto_verdict"] == "correct"
    assert normalized_claim["claim_text"] == "and attacks the c6-knight"
    assert verify_claim({"claim_text": "The black king is on e8."}, b, move, ctx)["auto_verdict"] == "correct"
    assert verify_claim({"claim_text": "The black king is on d8."}, b, move, ctx)["auto_verdict"] == "incorrect"
    assert verify_claim({"claim_text": "threatens to capture the knight on c6", "source_subject": "The bishop"}, b, move, ctx)["auto_verdict"] == "unknown"

    b = chess.Board("4rrk1/1p3pp1/2p2b1p/p1Pn4/4b3/P3PN1P/1P1B1KP1/1BRR4 w - - 5 25")
    move = b.parse_san("Bxe4")
    result = verify_claim({"claim_text": "or targeting other black pieces", "source_subject": "The bishop on c4"}, b, move, ctx)
    assert result["auto_verdict"] == "incorrect" and "c4" in result["auto_detail"], result
    assert verify_claim({"claim_text": "The bishop on c4 captures the knight on e4"}, b, move, ctx)["auto_verdict"] == "incorrect"
    assert verify_claim({"claim_text": "The bishop on b1 captures the bishop on e4"}, b, move, ctx)["auto_verdict"] == "correct"
    b = chess.Board("4k3/8/2n5/8/4p3/8/8/1B2K3 w - - 0 1")
    move = b.parse_san("Bxe4")
    assert verify_claim({"claim_text": "attacks the knight on c6", "source_subject": "The bishop on b1"}, b, move, ctx)["auto_verdict"] == "correct"
    assert verify_claim({"claim_text": "attacks the knight on c6", "source_subject": "The rook on b1"}, b, move, ctx)["auto_verdict"] == "incorrect"

    b = chess.Board("rn3rk1/pp2qp1p/2p2p2/5b2/2NPp2P/8/P1Q2PP1/1R2KB1R b K - 0 16")
    move = b.parse_san("Nd7")
    assert verify_claim({"claim_text": "The human player chooses Nd7"}, b, move, ctx)["auto_verdict"] == "correct"
    assert verify_claim({"claim_text": "moving the knight from g6 to d7"}, b, move, ctx)["auto_verdict"] == "incorrect"

    b = chess.Board("r5k1/4rpp1/2p2q1p/P2p1b2/1P1N1Q2/5P2/6PP/R2R2K1 b - - 2 25")
    move = b.parse_san("g5")
    assert verify_claim({"claim_text": "The move g5 advances the pawn"}, b, move, ctx)["auto_verdict"] == "correct"
    assert verify_claim({"claim_text": "The move g5 aims to open the g-file"}, b, move, ctx)["auto_verdict"] == "unknown"

    b = chess.Board("Q1b2rk1/p1p1bpp1/5q1p/4p3/2p5/8/PPPP1PPP/R1B1R1K1 b - - 2 14")
    move = b.parse_san("Bh3")
    assert verify_claim({"claim_text": "The human player chooses Bh3"}, b, move, ctx)["auto_verdict"] == "correct"
    print("Verifier regression tests passed.")


In [5]:
# Conservative extensions for checkable claim cores.
EXTEND_VERSION = "extend-v1.3"

# Only standalone discourse markers are filler; short chess assertions are not.
_PURE_FILLER = re.compile(
    r"^(?:overall|additionally|furthermore|moreover|however|finally|thus|therefore|"
    r"in addition|for example|in conclusion|as a result)[,!.?]*$", re.I
)


def is_pure_filler(claim):
    return bool(_PURE_FILLER.fullmatch(str(claim).strip()))


_EXTENSION_MODAL = re.compile(
    r"\b(?:might|may|could|potentially|possibly|perhaps|likely|aims?\s+to|"
    r"seeks?\s+to|intends?\s+to|plans?\s+to)\b", re.I
)
_HEDGE_CORE = re.compile(
    r"\b(?:might|may|could|potentially|possibly|perhaps|likely)\s+"
    r"(?P<core>(?:"
    r"(?:attack|target|eye|hit|pressure)(?:s)?\s+(?:the\s+)?(?:square\s+)?"
    r"(?:[a-h][1-8]|(?:(?:white|black)(?:'s)?\s+)?(?:king|queen|rook|bishop|knight|pawn)(?:\s+on\s+[a-h][1-8])?)"
    r"|(?:defend|protect|guard|support)(?:s)?\s+(?:the\s+|a\s+|an\s+)?"
    r"(?:(?:white|black)(?:'s)?\s+)?(?:king|queen|rook|bishop|knight|pawn)(?:\s+on\s+[a-h][1-8])?"
    r"))\b", re.I
)


def verify_hedged_core(claim_item, board, move, ctx):
    """Verify only a uniquely checkable attack/defence relation after stripping its hedge."""
    match = _HEDGE_CORE.search(normalize_claim_for_check(claim_item.get("claim_text")))
    if not match:
        return None
    result = verify_claim(
        {"claim_text": match.group("core"), "source_subject": claim_item.get("source_subject", "")},
        board, move, ctx, None,
    )
    if result["auto_verdict"] not in {"correct", "incorrect"} or result["auto_source"] != "python_chess":
        return None
    return verify_res(
        result["auto_verdict"], f"hedged_core_{result['auto_rule']}",
        f"(hedged factual core) {result['auto_detail']}", "python_chess", result["check_category"],
    )


def _connected_rooks(board, color):
    rooks = [sq for sq, piece in board.piece_map().items()
             if piece.color == color and piece.piece_type == chess.ROOK]
    if len(rooks) != 2:
        return None
    a, b = rooks
    if chess.square_rank(a) == chess.square_rank(b):
        step = 1 if chess.square_file(a) < chess.square_file(b) else -1
        between = (chess.square(f, chess.square_rank(a))
                   for f in range(chess.square_file(a) + step, chess.square_file(b), step))
    elif chess.square_file(a) == chess.square_file(b):
        step = 1 if chess.square_rank(a) < chess.square_rank(b) else -1
        between = (chess.square(chess.square_file(a), r)
                   for r in range(chess.square_rank(a) + step, chess.square_rank(b), step))
    else:
        return False
    return all(board.piece_at(sq) is None for sq in between)


def _line_opening(board, after, color, line_squares, slider_types):
    """Return evidence only when a stationary friendly slider gains line attacks."""
    line_squares = set(line_squares)
    for sq in line_squares:
        before_piece, after_piece = board.piece_at(sq), after.piece_at(sq)
        if (before_piece is None or after_piece is None
                or before_piece.color != color or after_piece.color != color
                or before_piece.piece_type not in slider_types
                or after_piece.piece_type != before_piece.piece_type):
            continue
        before_count = sum(target in board.attacks(sq) for target in line_squares)
        after_count = sum(target in after.attacks(sq) for target in line_squares)
        if after_count > before_count:
            return chess.square_name(sq), before_count, after_count
    return None


def _side_attacks(board, color):
    attacked = set()
    for sq, piece in board.piece_map().items():
        if piece.color == color:
            attacked.update(board.attacks(sq))
    return attacked


def verify_strategic_auto(claim_item, board, after, move, mover):
    """Auto-verify only direct, conservative board facts; leave strategic inference unresolved."""
    text = normalize_claim_for_check(claim_item.get("claim_text"))
    subject = str(claim_item.get("source_subject") or "")
    full_claim = normalize_claim_for_check(f"{subject} {text}".strip())
    if _NEG.search(full_claim) or _EXTENSION_MODAL.search(full_claim):
        return None

    if re.search(r"\bconnect(?:s|ed|ing)?\s+(?:the\s+)?rooks?\b", full_claim, re.I):
        was_connected = _connected_rooks(board, mover)
        is_connected = _connected_rooks(after, mover)
        if was_connected is False and is_connected is True:
            return verify_res("correct", "connects_rooks", "the move connects the two friendly rooks",
                              "python_chess", "Board comprehension")

    opened = re.search(
        r"\bopen(?:s|ed|ing)?\s+(?:the\s+)?(?P<line>(?:[a-h]\s*-\s*file|a1\s*-\s*h8|a8\s*-\s*h1))\b",
        full_claim, re.I,
    )
    if opened:
        target = re.sub(r"\s+", "", opened.group("line")).lower()
        if re.fullmatch(r"[a-h]-file", target):
            file_index = ord(target[0]) - ord("a")
            line = {chess.square(file_index, rank) for rank in range(8)}
            evidence = _line_opening(board, after, mover, line, {chess.ROOK, chess.QUEEN})
        else:
            line = ({chess.square(i, i) for i in range(8)} if target == "a1-h8"
                    else {chess.square(i, 7 - i) for i in range(8)})
            evidence = _line_opening(board, after, mover, line, {chess.BISHOP, chess.QUEEN})
        if evidence:
            square, before_count, after_count = evidence
            return verify_res("correct", "opens_line",
                              f"friendly slider on {square} gains line attacks ({before_count} to {after_count})",
                              "python_chess", "Board comprehension")

    central = {chess.D4, chess.E4, chess.D5, chess.E5}
    specific_center = re.search(r"\bcontrol(?:s|led|ling)?\s+(?:the\s+)?(?P<square>[de][45])\b", full_claim, re.I)
    generic_center = re.search(r"\bcontrol(?:s|led|ling)?\s+(?:the\s+)?(?:center|centre)\b", full_claim, re.I)
    subject_names_piece = bool(re.search(r"\b(?:king|queen|rook|bishop|knight|pawn)\b", subject, re.I))
    text_names_piece = bool(re.search(r"\b(?:king|queen|rook|bishop|knight|pawn)\b", text, re.I))
    if not subject_names_piece and not text_names_piece:
        side_match = re.search(r"\b(white|black)(?:'s)?\s+(?:pieces?\s+)?control(?:s|led|ling)?\b", full_claim, re.I)
        color = COLORS[side_match.group(1).lower()] if side_match else mover
        before_attacks, after_attacks = _side_attacks(board, color), _side_attacks(after, color)
        if specific_center:
            square = chess.parse_square(specific_center.group("square").lower())
            if square in after_attacks and square not in before_attacks:
                return verify_res("correct", "controls_center_square",
                                  f"the side newly attacks {chess.square_name(square)} after the move",
                                  "python_chess", "Board comprehension")
        elif generic_center and central.issubset(after_attacks) and not central.issubset(before_attacks):
            return verify_res("correct", "controls_center",
                              "the side attacks all four central squares after the move",
                              "python_chess", "Board comprehension")

    centralizes = re.search(
        r"\bcentraliz(?:e|es|ed|ing)\s+(?:the\s+)?(?:(?P<side>white|black)(?:'s)?\s+)?"
        r"(?P<piece>king|queen|rook|bishop|knight)\b", full_claim, re.I,
    )
    if centralizes:
        moved = board.piece_at(move.from_square)
        side = COLORS[centralizes.group("side").lower()] if centralizes.group("side") else mover
        if (moved is not None and moved.color == side
                and moved.piece_type == PIECE_TYPES[centralizes.group("piece").lower()]
                and move.from_square not in central and move.to_square in central):
            return verify_res("correct", "centralizes_piece",
                              f"the {chess.piece_name(moved.piece_type)} moves to {chess.square_name(move.to_square)}",
                              "python_chess", "Board comprehension")

    return None


_PIN_TARGET = re.compile(
    r"\bpin(?:s|ned|ning)?\s+(?:the\s+)?(?:(?P<side>white|black)(?:'s)?\s+)?"
    r"(?P<piece>king|queen|rook|bishop|knight|pawn)\s+(?:on|at)\s+(?P<square>[a-h][1-8])\b"
    r"|\b(?:(?P<side_after>white|black)(?:'s)?\s+)?(?P<piece_after>king|queen|rook|bishop|knight|pawn)"
    r"\s+(?:on|at)\s+(?P<square_after>[a-h][1-8])\s+(?:is|was|remains)\s+pinned\b", re.I,
)


def detect_hallucination(claim_item, board, after, move, mover):
    """Reject a pinned-piece claim only when its explicit target square/type is wrong."""
    text = normalize_claim_for_check(claim_item.get("claim_text"))
    subject = str(claim_item.get("source_subject") or "")
    full_claim = normalize_claim_for_check(f"{subject} {text}".strip())
    if _NEG.search(full_claim) or _EXTENSION_MODAL.search(full_claim) or re.search(r"\bbefore\s+(?:the\s+)?move\b", full_claim, re.I):
        return None
    match = _PIN_TARGET.search(full_claim)
    if not match:
        return None
    side = match.group("side") or match.group("side_after")
    piece_name = match.group("piece") or match.group("piece_after")
    square_name = match.group("square") or match.group("square_after")
    target = after.piece_at(chess.parse_square(square_name.lower()))
    if (target is None or target.piece_type != PIECE_TYPES[piece_name.lower()]
            or (side and target.color != COLORS[side.lower()])):
        actual = "nothing" if target is None else f"a {'white' if target.color == chess.WHITE else 'black'} {chess.piece_name(target.piece_type)}"
        return verify_res("incorrect", "pin_target_mismatch",
                          f"{square_name} contains {actual}, not the claimed {piece_name}",
                          "python_chess", "Tactical reasoning")
    return None


def run_extended_self_tests():
    assert is_pure_filler("As a result")
    assert not is_pure_filler("preparing for future attacks")
    assert not is_pure_filler("the pawn advances")

    board = chess.Board("4k3/8/2n5/8/4p3/8/8/4K2B w - - 0 1")
    move = board.parse_san("Bxe4")
    ctx = {}
    result = verify_hedged_core(
        {"claim_text": "might attack the knight on c6", "source_subject": "The bishop"},
        board, move, ctx,
    )
    assert result and result["auto_verdict"] == "correct", result
    square_core = verify_hedged_core(
        {"claim_text": "might attack d5", "source_subject": "The bishop"}, board, move, ctx,
    )
    assert square_core and square_core["auto_verdict"] == "correct", square_core
    after = board.copy(); after.push(move)
    pin = detect_hallucination(
        {"claim_text": "pins the knight on e4", "source_subject": "The bishop"},
        board, after, move, board.turn,
    )
    assert pin and pin["auto_verdict"] == "incorrect", pin

    board = chess.Board("4k3/8/8/8/8/8/4K3/R1B4R w - - 0 1")
    move = board.parse_san("Bb2")
    after = board.copy(); after.push(move)
    result = verify_strategic_auto({"claim_text": "The move connects the rooks"}, board, after, move, board.turn)
    assert result and result["auto_verdict"] == "correct", result

    board = chess.Board("4k3/8/8/8/8/8/1P6/1R2K3 w - - 0 1")
    move = board.parse_san("b4")
    after = board.copy(); after.push(move)
    result = verify_strategic_auto({"claim_text": "The move opens the b-file"}, board, after, move, board.turn)
    assert result and result["auto_verdict"] == "correct", result

    board = chess.Board("4k3/8/8/8/2N5/8/8/4K3 w - - 0 1")
    move = board.parse_san("Ne5")
    after = board.copy(); after.push(move)
    result = verify_strategic_auto({"claim_text": "The move centralizes the knight"}, board, after, move, board.turn)
    assert result and result["auto_verdict"] == "correct", result

    board = chess.Board("r5k1/4rpp1/2p2q1p/P2p1b2/1P1N1Q2/5P2/6PP/R2R2K1 b - - 2 25")
    move = board.parse_san("g5")
    after = board.copy(); after.push(move)
    assert verify_strategic_auto({"claim_text": "The move aims to open the g-file"}, board, after, move, board.turn) is None
    assert verify_strategic_auto({"claim_text": "The move improves the rook's activity"}, board, after, move, board.turn) is None
    print("Extended verifier regression tests passed.")


## 0. Run control

In [6]:
RUN_SPLITTER = True          # True: call the splitter LLM for explanations that are not in the cache yet (Ollama must be running)
RUN_LIVE_STOCKFISH = True    # True: evaluate alternative moves named in explanations with Stockfish (depth/threads/hash from config.py)

## 1. Verify the Step 3 inputs (hash chain and CPU settings)

In [7]:
pilot_manifest = json.loads((P1 / "pilot_manifest.json").read_text(encoding="utf-8"))
pilot_for_llm = pd.read_csv(P1 / "pilot_for_llm.csv", dtype=str)
pilot_key = pd.read_csv(P1 / "pilot_key.csv", dtype={"pilot_id": str, "position_id": str})
if sha256_file(P1 / "pilot_for_llm.csv") != pilot_manifest["llm_input_sha256"]:
    warn("pilot_for_llm.csv differs from pilot_manifest.json")

rows = []
for m in MODELS:
    d = S3 / slug(m)
    man = json.loads((d / "run_manifest.json").read_text(encoding="utf-8"))
    raw = (d / "generations.jsonl").read_bytes()
    gen_ok = man.get("generations_sha256") in (hashlib.sha256(raw).hexdigest(), hashlib.sha256(raw.replace(b"\r\n", b"\n")).hexdigest())
    o = man["settings"]["options"]
    prefix = config.LLM_PILOT_MODEL_ID_PREFIXES.get(m)
    rows.append(dict(model=m, step=man.get("step"), digest=man["model_digest"][:12], digest_ok=(prefix is None or man["model_digest"].removeprefix("sha256:").startswith(prefix)),
                     ollama=man["ollama_version"], num_gpu=o["num_gpu"], threads=o["num_thread"], temp=o["temperature"], seed=o["seed"],
                     num_ctx=o["num_ctx"], num_predict=o["num_predict"], repeats=man["repeats_per_condition"],
                     done=f'{man["completed_generations"]}/{man["expected_generations"]}', generations_hash_ok=gen_ok,
                     input_ok=man["settings"]["input_csv_sha256"] == pilot_manifest["llm_input_sha256"], clean_git=not man["git_uncommitted_changes"]))
env = pd.DataFrame(rows)
print(env.to_string(index=False))
if (env.step != "step3_llm_pilot").any(): warn("a run manifest is not from step3_llm_pilot")
if (env.num_gpu != 0).any(): warn("a model was not run CPU-only (num_gpu != 0)")
for col in ("threads", "ollama", "seed", "temp"):
    if env[col].nunique() > 1: warn(f"{col} differs between models: {env[col].tolist()}")
if not env.digest_ok.all(): warn("a model digest differs from config.LLM_PILOT_MODEL_ID_PREFIXES")
if not env.input_ok.all(): warn("a Step 3 run used a different pilot_for_llm.csv")
if not env.generations_hash_ok.all(): warn("generations.jsonl differs from the hash in its run_manifest.json")
if (env.repeats < 2).all(): warn("only 1 repeat per condition: run-to-run determinism on CPU is NOT yet tested")

           model            step       digest  digest_ok  ollama  num_gpu  threads  temp  seed  num_ctx  num_predict  repeats  done  generations_hash_ok  input_ok  clean_git
      qwen2.5:7b step3_llm_pilot 845dbda0ea48       True 0.40.18        0        8   0.0    46     2048          160        2 60/60                 True      True      False
  deepseek-r1:8b step3_llm_pilot 6995872bfe4c       True 0.40.18        0        8   0.0    46     4096         1500        2 60/60                 True      True      False
phi4-mini:latest step3_llm_pilot 78fad5d182a7       True 0.40.18        0        8   0.0    46     2048          160        2 60/60                 True      True      False
     llama3.1:8b step3_llm_pilot 46e0c10c039e       True 0.40.18        0        8   0.0    46     2048          160        2 60/60                 True      True      False


## 2. Load generations, separate the verdict line, quality control

In [8]:
gen = pd.DataFrame([json.loads(l) for m in MODELS for l in (S3 / slug(m) / "generations.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()])
if gen.duplicated(KEY).any(): raise ValueError("duplicate (model, condition, pilot_id, repeat) rows in the generations")
expected = len(MODELS) * len(CONDITIONS) * len(pilot_for_llm) * REPEATS
if len(gen) != expected: warn(f"{len(gen)} generations found, expected {expected}")
gen["truncated"], gen["think_unterminated"] = gen.truncated.astype(bool), gen.think_unterminated.astype(bool)
parsed = gen.text.map(split_verdict)
gen["explanation"], gen["verdict"] = parsed.str[0], parsed.str[1]
gen["words"] = gen.explanation.str.split().str.len()
gen["sentences"] = gen.explanation.map(lambda t: len(split_sentences(t)))
gen["usable"] = ~gen.truncated & ~gen.think_unterminated & gen.explanation.str.strip().ne("")

a = gen.loc[gen.condition == "A", "prompt"]
if a.str.contains(r"stockfish|centipawn|\bcp\b|principal|engine|evaluation", case=False, regex=True).any():
    warn("a Condition A prompt contains engine information")
if (gen.groupby(["condition", "pilot_id"]).prompt.nunique() > 1).any():
    warn("different models received different prompts for the same position and condition")
for c in CONDITIONS[1:]:
    print(f"--- sample Condition {c} prompt ---\n{gen.loc[gen.condition == c, 'prompt'].iloc[0][:800]}\n")

qc = gen.groupby(["model", "condition"]).agg(n=("usable", "size"), usable=("usable", "sum"), truncated=("truncated", "sum"),
        verdict_missing=("verdict", lambda s: int(s.isna().sum())), mean_words=("words", "mean"), mean_sentences=("sentences", "mean"),
        over_3_sentences=("sentences", lambda s: int((s > 3).sum())), mean_seconds=("seconds", "mean")).round(1).reset_index()
print(qc.to_string(index=False))
excluded = gen.loc[~gen.usable, KEY + ["done_reason", "words", "truncated", "think_unterminated"]]
if len(excluded): warn(f"{len(excluded)} unusable generation(s) excluded (see excluded_generations.csv)")
if gen.verdict.isna().any(): warn(f"{int(gen.verdict.isna().sum())} generation(s) without a parsable 'Verdict:' line (format failure, kept in the data)")

--- sample Condition B prompt ---
Chess position (FEN): 4rrk1/1p3pp1/2p2b1p/p1Pn4/4b3/P3PN1P/1P1B1KP1/1BRR4 w - - 5 25
White to move.
The move played by the human is Bxe4 (standard algebraic notation).

Explain why a human might choose this move. Describe what it does and what it changes in the position. Use at most 3 short sentences and refer to specific pieces and squares.
Then add a last line in exactly this form: Verdict: good, inaccurate or bad (your own judgement of the move).
Stockfish evaluation before the move (from White's point of view; positive = good for White): -15 cp.
Stockfish evaluation after the move (from White's point of view; positive = good for White): -139 cp.


--- sample Condition C prompt ---
Chess position (FEN): 4rrk1/1p3pp1/2p2b1p/p1Pn4/4b3/P3PN1P/1P1B1KP1/1BRR4 w - - 5 25
White to move.
The move played by the human is Bxe4 (standard algebraic notation).

Explain why a human might choose this move. Describe what it does and what it changes in the position. 

## 3. Reference data of the pilot positions (hidden labels and stored Stockfish results, used only by the checker)

In [9]:
sf = pd.read_csv(SF / "positions_stockfish.csv", dtype={"position_id": str})
need = ["position_id", "fen", "eval_before_cp", "eval_after_cp", "engine_best_pv_san", "pv_after_san", "human_move_is_engine_best"]
ref = (pilot_for_llm.merge(pilot_key[["pilot_id", "position_id", "kind", "stockfish_cp_loss", "engine_best_san"]], on="pilot_id", how="left")
       .merge(sf[need].rename(columns={"fen": "fen_sf"}), on="position_id", how="left"))
assert len(ref) == len(pilot_for_llm) and ref.eval_before_cp.notna().all(), "pilot positions not found in positions_stockfish.csv"
assert (ref.fen == ref.fen_sf).all(), "FEN differs between pilot_for_llm.csv and positions_stockfish.csv"
ref = ref.set_index("pilot_id")


def make_ctx(r, condition):
    sgn = 1 if r.side_to_move.lower() == "white" else -1
    return dict(label=r.kind, cp_loss=int(r.stockfish_cp_loss), eval_before_mover=sgn * int(r.eval_before_cp), eval_after_mover=sgn * int(r.eval_after_cp),
                eval_after_white=int(r.eval_after_cp), best_san=r.engine_best_san, best_pv=json.loads(r.engine_best_pv_san),
                pv_after=json.loads(r.pv_after_san), human_is_best=bool(r.human_move_is_engine_best),
                provided_evals=(sgn * int(r.eval_before_cp), sgn * int(r.eval_after_cp)) if condition in ("B", "C") else ())


print(ref[["kind", "move_to_explain_san", "stockfish_cp_loss", "eval_before_cp", "eval_after_cp", "human_move_is_engine_best"]].to_string())

          kind move_to_explain_san  stockfish_cp_loss  eval_before_cp  eval_after_cp  human_move_is_engine_best
pilot_id                                                                                                       
P001       bad                Bxe4                124             -15           -139                      False
P002      good                 Bg6                  0            -329           -356                       True
P003       bad                  g5                467             254            721                      False
P004       bad                 Rc1                131            -473           -604                      False
P005       bad                 Nd7                213            -246            -33                      False
P006      good                Rxf4                  0            -288           -288                       True
P007       bad                 Bh3                144               7            151                    

## 4. Split into claims (splitter LLM + validation + cache)

In [10]:
SPLIT_SETTINGS = dict(model=config.SPLITTER_MODEL, temperature=0.0, seed=config.SEED_LLM, num_gpu=0, num_ctx=config.SPLITTER_NUM_CTX,
                      num_predict=config.SPLITTER_MAX_TOKENS, version=SPLITTER_VERSION)
SPLIT_OPTIONS = dict(temperature=0.0, seed=config.SEED_LLM, num_gpu=0, num_thread=NUM_THREADS, num_ctx=config.SPLITTER_NUM_CTX, num_predict=config.SPLITTER_MAX_TOKENS)
CACHE = OUT / "splitter_cache.jsonl"


def eid_of(g): return f"{slug(g.model)}|{g.condition}|r{g.repeat}|{g.pilot_id}"


def chat_fn(text):
    if not RUN_SPLITTER:
        raise RuntimeError("An explanation is not in the splitter cache. Set RUN_SPLITTER = True (Ollama running) in the run-control cell and run again.")
    return ollama_chat(OLLAMA_URL, config.SPLITTER_MODEL, text, SPLIT_OPTIONS)


use = gen[gen.usable]
texts = {eid_of(g): g.explanation for g in use.itertuples(index=False)}
splits = split_all(texts, chat_fn, SPLIT_SETTINGS, CACHE)
src = pd.Series({k: v[1]["split_source"] for k, v in splits.items()}).value_counts()
print(f"explanations split: {len(splits)} | by source: {src.to_dict()}")
fallbacks = {k: v[1]["split_note"] for k, v in splits.items() if v[1]["split_source"] != "llm"}
if fallbacks: warn(f"{len(fallbacks)} explanation(s) fell back to the rule-based splitter (first reasons: {list(fallbacks.values())[:3]})")
digest = None
if RUN_SPLITTER:
    tags = json.loads(urllib.request.urlopen(OLLAMA_URL + "/api/tags").read())["models"]
    digest = next((m["digest"] for m in tags if m["name"] == config.SPLITTER_MODEL), None)
    ollama_version = json.loads(urllib.request.urlopen(OLLAMA_URL + "/api/version").read()).get("version")
else:
    ollama_version = None

Claim split progress: 226 explanations; 205 cached text(s) match the current splitter settings.
[1/226] Processed cached response: llm, 3 claim(s), 0.00s for current item.
[5/226] Processed cached response: rule_fallback, 3 claim(s), 0.00s for current item.
[10/226] Processed cached response: rule_fallback, 3 claim(s), 0.00s for current item.
[15/226] Processed cached response: llm, 4 claim(s), 0.00s for current item.
[20/226] Processed cached response: llm, 4 claim(s), 0.00s for current item.
[25/226] Processed cached response: rule_fallback, 4 claim(s), 0.00s for current item.
[30/226] Processed cached response: rule_fallback, 4 claim(s), 0.00s for current item.
[35/226] Processed cached response: rule_fallback, 3 claim(s), 0.00s for current item.
[40/226] Processed cached response: rule_fallback, 4 claim(s), 0.00s for current item.
[45/226] Processed cached response: rule_fallback, 3 claim(s), 0.00s for current item.
[50/226] Processed cached response: rule_fallback, 4 claim(s), 0.0

## 5. Verify, classify, route (verify-v8.4 + extend-v1.3)

The verifier uses the splitter's exact quote and separately stored subject without changing either. It checks explicit board relations against a uniquely identified source piece, leaves ambiguous relations and unsupported threats unknown, checks complete square assertions in the relevant position, and applies the existing chess and Stockfish checks to remaining claims.

Routes: `excluded_framing` (restates the prompt, no content; not a factual claim), `auto_python_chess` / `auto_stockfish_stored` / `auto_stockfish_live` / `auto_label` (a rule decided it),
`unverifiable_strategic` (positional judgement no program can settle), `unverifiable_hedged` (could/might/aims to), `manual`.

In [11]:
engine = None
if RUN_LIVE_STOCKFISH:
    if sys.platform == "win32":
        import asyncio
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    sf_path = Path(config.STOCKFISH_PATH) if Path(config.STOCKFISH_PATH).is_absolute() else PROJECT / config.STOCKFISH_PATH
    engine = LiveEngine(sf_path, config.STOCKFISH_DEPTH, config.STOCKFISH_THREADS, config.STOCKFISH_HASH_MB, OUT / "live_stockfish_cache.jsonl")
    print("live engine:", engine.version)


def route(r):
    if r.is_framing or r.is_pure_filler: return "excluded_framing"
    if r.auto_verdict != "unknown": return "auto_" + r.auto_source
    if r.is_strategic: return "unverifiable_strategic"
    if r.hedged: return "unverifiable_hedged"
    return "manual"


def run_extraction(df, engine=None):
    out = []
    for g in df.sort_values(KEY).itertuples(index=False):
        r = ref.loc[g.pilot_id]
        board = chess.Board(r.fen); move = board.parse_san(r.move_to_explain_san)
        ctx = make_ctx(r, g.condition)
        eid = eid_of(g)
        claims_i, info = splits[eid]
        items = [dict(c, claim_kind="explanation") for c in claims_i]
        if g.verdict: items.append(dict(sentence_idx=-1, clause_idx=len(items), sentence=f"Verdict: {g.verdict}", claim_text=f"Verdict: {g.verdict}", source_quote=f"Verdict: {g.verdict}", source_subject="", hedged=False, claim_kind="verdict"))
        for i, c in enumerate(items):
            if c["claim_kind"] == "verdict":
                k = dict(category="Quality evaluation", category_rule="verdict_line", also_matches="", is_strategic=False, review_category=False, mentions_engine=False)
                p = verdict_check(g.verdict, ctx["label"])
                pure_filler = False
            else:
                k = classify(c["claim_text"])
                if rd_is_vague(c["claim_text"]): k = dict(k, is_strategic=True)
                pure_filler = is_pure_filler(c["claim_text"]) or rd_is_filler(c["claim_text"])
                pv = rd_verify_verdict_text(c["claim_text"], ctx)
                if pv is not None:
                    p, pure_filler = pv, False
                elif pure_filler:
                    p = verify_res("unknown", "pure_filler", "standalone discourse marker; no factual claim", "none")
                else:
                    after = board.copy(); after.push(move); mover = board.turn
                    cv = dict(c, claim_text=rd_resolve(c), claim_text_orig=c["claim_text"])
                    if rd_is_intent(c["claim_text"]) and not c.get("hedged"):
                        c = dict(c, hedged=True)   # "prepares to ..." is a plan, not a board fact
                    p = rd_verify_echo(cv, ctx)
                    if p is None:
                        p = detect_hallucination(c, board, after, move, mover)
                    if p is None:
                        p = verify_strategic_auto(c, board, after, move, mover)
                    if p is None and c.get("hedged"):
                        p = verify_hedged_core(c, board, move, ctx)
                    if p is None:
                        p = verify_claim(c, board, move, ctx, engine)
                        if p["auto_verdict"] == "unknown" and cv["claim_text"] != c["claim_text"] and not k["is_strategic"]:
                            p = verify_claim(cv, board, move, ctx, engine)
                    if p["auto_verdict"] == "unknown":
                        q = rd_verify(cv, board, after, move, mover, ctx, engine)
                        if q is not None:
                            p = q
            cat = p["check_category"] or k["category"]
            out.append(dict(claim_id=f"{eid}|{i:02d}", explanation_id=eid, model=g.model, condition=g.condition, repeat=g.repeat, pilot_id=g.pilot_id,
                            split_source=info["split_source"], **c, **{x: k[x] for x in ("category_rule", "also_matches", "is_strategic", "review_category", "mentions_engine")},
                            category=cat, category_source="check" if p["check_category"] else "keywords",
                            is_framing=(c["claim_kind"] == "explanation" and is_framing(c["claim_text"])),
                            is_pure_filler=pure_filler, **p))
    df = pd.DataFrame(out)
    df["route"] = df.apply(route, axis=1)
    return df


run_verifier_self_tests()
run_extended_self_tests()
claims = run_extraction(use, engine)
again = run_extraction(use, engine)
selftest = hashlib.sha256(csv_bytes(claims)).hexdigest() == hashlib.sha256(csv_bytes(again)).hexdigest()
print(f"{len(claims)} claims from {claims.explanation_id.nunique()} explanations | determinism self-test (from cache): {selftest}")
if not selftest: warn("extraction/verification is not deterministic")
if engine is not None: engine.close()
print(claims.sample(min(15, len(claims)), random_state=config.SEED_ANNOTATION)[["model", "condition", "category", "route", "auto_rule", "claim_text"]].to_string(index=False, max_colwidth=60))


live engine: Stockfish 19
Verifier regression tests passed.
Extended verifier regression tests passed.
1301 claims from 226 explanations | determinism self-test (from cache): True
           model condition            category                  route             auto_rule                                                   claim_text
  deepseek-r1:8b         B Board comprehension                 manual   claim_needs_context Bxe4 sacrifices the bishop on c4 to attack the black king...
      qwen2.5:7b         C Board comprehension                 manual   claim_needs_context          preventing any immediate attack on the knight on b1
phi4-mini:latest         C               Other unverifiable_strategic              no_check The move Rb1 aims to reposition the rook to a more active...
phi4-mini:latest         B  Tactical reasoning                 manual              no_check                 which is now vulnerable to potential attacks
     llama3.1:8b         A  Quality evaluation         

## 6. Summaries - how much is left for manual work?

Direct board facts are auto-routed only when a conservative rule establishes them. The extension leaves strategic activity claims and unsupported/hedged statements unresolved; it does not treat short claims as filler.

In [12]:
body = claims[~claims.is_framing]
per_expl = claims.groupby(["model", "condition", "explanation_id"]).size().rename("n_claims").reset_index()
stats = per_expl.groupby(["model", "condition"]).n_claims.agg(explanations="count", mean="mean", sd="std", min="min", max="max").round(2).reset_index()
routes = claims.groupby(["model", "condition", "route"]).size().unstack(fill_value=0).reset_index()
share = (body.groupby("route").size() / len(body)).round(3).rename("share_of_factual_claims").reset_index()
by_rule = claims[claims.route.str.startswith("auto_")].groupby(["auto_source", "auto_rule", "auto_verdict"]).size().unstack(fill_value=0).reset_index()
cats = claims.groupby(["model", "condition", "category"]).size().unstack(fill_value=0).reindex(columns=list(CATEGORIES), fill_value=0).reset_index()
cat_src = claims.groupby("category_source").size()
for name, t in (("claims per explanation", stats), ("route by model and condition", routes), ("share of factual claims by route", share),
                ("automatic checks by rule", by_rule), ("categories", cats)):
    print(f"\n== {name} =="); print(t.to_string(index=False))
print("\ncategory assigned by:", cat_src.to_dict())
live_off = int((claims.auto_rule == "live_engine_off").sum())
if live_off: print(f"\n{live_off} claim(s) name an alternative move that live Stockfish could judge: set RUN_LIVE_STOCKFISH = True")
manual_share = float((body.route == "manual").mean())
print(f"\nMANUAL share of factual claims: {manual_share:.1%}  ({int((body.route == 'manual').sum())} of {len(body)})")


== claims per explanation ==
           model condition  explanations  mean   sd  min  max
  deepseek-r1:8b         A             7  5.86 1.35    4    8
  deepseek-r1:8b         B            20  5.65 1.09    4    7
  deepseek-r1:8b         C            20  6.30 1.81    4   12
     llama3.1:8b         A            19  6.95 1.13    5    9
     llama3.1:8b         B            20  6.70 0.98    5    8
     llama3.1:8b         C            20  6.50 0.76    5    8
phi4-mini:latest         A            20  5.15 1.23    4    8
phi4-mini:latest         B            20  6.25 0.97    5    8
phi4-mini:latest         C            20  5.75 1.37    4   10
      qwen2.5:7b         A            20  4.35 0.88    3    6
      qwen2.5:7b         B            20  4.90 0.91    3    6
      qwen2.5:7b         C            20  4.85 1.14    3    8

== route by model and condition ==
           model condition  auto_label  auto_python_chess  auto_stockfish_stored  excluded_framing  manual  unverifiable_hedged 

## 7. Audit and annotation files

* `splitting_audit.csv`: seeded sample of explanations next to their claims. Count missed / spurious / badly split claims by hand (also report how often the splitter fell back to rules).
* `annotation_sheet_blind.csv`: all `manual` claims **plus** a seeded `AUTO_AUDIT_FRACTION` sample of auto-verified claims, shuffled; the annotator cannot tell which is which and sees no model, condition, route or proposed verdict. It includes the associated source sentence, the exact quote and subject when the LLM splitter supplied them, and the constructed claim text. Rule-based fallback claims have no source quote. `annotation_key.csv` maps back.
  Compare the annotator's verdicts with the automatic ones for the audit sample: that is the checker's precision, reported per rule.
* `annotation_second_subset.csv`: stratified (condition x category) `ANNOTATION_FRACTION` of the sheet for the second annotator.

In [13]:
rng = random.Random(config.SEED_ANNOTATION)
audit_ids = sorted(rng.sample(sorted(claims.explanation_id.unique()), min(AUDIT_N, claims.explanation_id.nunique())))
audit = []
for eid in audit_ids:
    items = claims[(claims.explanation_id == eid) & (claims.claim_kind == "explanation")].claim_text.tolist()
    audit.append(dict(explanation_id=eid, original_text=texts[eid], extracted="\n".join(f"{i + 1}. {t}" for i, t in enumerate(items)),
                      split_source=splits[eid][1]["split_source"], n_extracted=len(items), n_missed="", n_spurious="", n_badly_split="", notes=""))
audit = pd.DataFrame(audit)

autos = sorted(claims[claims.route.str.startswith("auto_")].claim_id)
auto_sample = set(random.Random(f"{config.SEED_ANNOTATION}:auto").sample(autos, math.ceil(config.AUTO_AUDIT_FRACTION * len(autos)))) if autos else set()
all_labelled = claims[(claims.route == "manual") | claims.claim_id.isin(auto_sample)].copy()
all_labelled["dedup_key"] = all_labelled.pilot_id.astype(str) + "||" + all_labelled.sentence.map(norm) + "||" + all_labelled.claim_text.map(norm)
to_label = all_labelled.sort_values("claim_id").drop_duplicates("dedup_key")   # identical claim in the same position and sentence: label once
n_instances = all_labelled.groupby("dedup_key").size()
audit_keys = set(all_labelled.loc[all_labelled.claim_id.isin(auto_sample), "dedup_key"])
order = sorted(to_label.claim_id); random.Random(f"{config.SEED_ANNOTATION}:blind").shuffle(order)
blind_id = {cid: f"C{i + 1:04d}" for i, cid in enumerate(order)}
c2 = to_label.assign(blind_id=to_label.claim_id.map(blind_id)).sort_values("blind_id")
c2 = c2.merge(ref[["fen", "side_to_move", "move_to_explain_san"]], left_on="pilot_id", right_index=True)
sheet = c2[["blind_id", "fen", "side_to_move", "move_to_explain_san", "sentence", "source_quote", "source_subject", "claim_text"]].assign(verdict="", category_manual="", notes="")
instance_map = all_labelled[["claim_id", "dedup_key"]].assign(blind_id=all_labelled.dedup_key.map(dict(zip(c2.dedup_key, c2.blind_id)))).drop(columns="dedup_key")
sub_ids, r2 = [], random.Random(f"{config.SEED_ANNOTATION}:second")
for _, g in c2.groupby(["condition", "category"]):
    sub_ids += r2.sample(sorted(g.claim_id), max(1, math.ceil(config.ANNOTATION_FRACTION * len(g))))
key = c2[["blind_id", "claim_id", "explanation_id", "model", "condition", "repeat", "pilot_id", "claim_kind", "category", "category_source", "category_rule", "route",
          "auto_verdict", "auto_source", "auto_rule", "auto_detail", "mentions_engine"]].assign(
    in_auto_audit=c2.dedup_key.isin(audit_keys).values, n_instances=c2.dedup_key.map(n_instances).values, second_annotator=c2.claim_id.isin(set(sub_ids)).values)
second = sheet[key.second_annotator.values]
print(f"annotation sheet: {len(sheet)} claims ({int((~key.in_auto_audit).sum())} manual + {int(key.in_auto_audit.sum())} auto-audit) | second-annotator subset: {len(second)}")

annotation sheet: 377 claims (326 manual + 51 auto-audit) | second-annotator subset: 82


## 8. Write the outputs and the manifest

In [14]:
tables = {"claims_all.csv": claims, "generations_qc.csv": qc, "excluded_generations.csv": excluded, "summary_claims_per_explanation.csv": stats,
          "summary_routes.csv": routes, "summary_route_share.csv": share, "summary_auto_rules.csv": by_rule, "summary_categories.csv": cats,
          "splitting_audit.csv": audit, "annotation_sheet_blind.csv": sheet, "annotation_key.csv": key, "annotation_second_subset.csv": second, "annotation_instance_map.csv": instance_map}
for name, df in tables.items(): (OUT / name).write_bytes(csv_bytes(df))
h = hashlib.sha256()
for name in sorted(tables): h.update(name.encode()); h.update((OUT / name).read_bytes())
manifest = dict(step="step4_claims", splitter=dict(settings=SPLIT_SETTINGS, settings_sha256=settings_sha(SPLIT_SETTINGS), model_digest=digest, ollama_version=ollama_version,
                    cache_sha256=sha256_file(CACHE), explanations=len(splits), by_source={k: int(v) for k, v in src.items()}, fallback_reasons=fallbacks),
    lib_versions=dict(claims_lib=LIB_VERSION, claims_llm=SPLITTER_VERSION, claims_verify=VERIFY_VERSION, claims_extend=EXTEND_VERSION, claims_reduce=REDUCE_VERSION),
    helper_code_sha256=helper_cell_hashes(), notebook_source_sha256=notebook_code_sha256(),
    live_stockfish=dict(enabled=RUN_LIVE_STOCKFISH, version=None if engine is None else engine.version, depth=config.STOCKFISH_DEPTH, threads=config.STOCKFISH_THREADS,
                        hash_mb=config.STOCKFISH_HASH_MB, cache_sha256=sha256_file(OUT / "live_stockfish_cache.jsonl") if (OUT / "live_stockfish_cache.jsonl").exists() else None),
    month=MONTH, models=MODELS, conditions=CONDITIONS, repeats=REPEATS, generations=len(gen), usable_generations=int(gen.usable.sum()), claims=len(claims),
    manual_share_of_factual_claims=round(manual_share, 4), determinism_selftest_passed=selftest, flags=problems,
    pilot_manifest_sha256=sha256_file(P1 / "pilot_manifest.json"), step3_run_manifests_sha256={m: sha256_file(S3 / slug(m) / "run_manifest.json") for m in MODELS},
    seeds=dict(annotation=config.SEED_ANNOTATION, llm=config.SEED_LLM), annotation_fraction=config.ANNOTATION_FRACTION, auto_audit_fraction=config.AUTO_AUDIT_FRACTION,
    config=config.config_snapshot(), git_commit=git("rev-parse", "HEAD"), git_uncommitted_changes=bool(git("status", "--porcelain")), python=sys.version.split()[0],
    platform=platform.platform(), packages={p: metadata.version(p) for p in ("chess", "pandas")}, hashed_outputs=sorted(tables), results_sha256=h.hexdigest(),
    timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))
(OUT / "step4_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"results_sha256: {manifest['results_sha256']} | flags: {len(problems)} | uncommitted changes: {manifest['git_uncommitted_changes']}")

results_sha256: 9db871e24aae82dad4c99fcebf600c5c3d10b43f7a1d5b1bde49113a53d105c9 | flags: 3 | uncommitted changes: True
